In [31]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
1,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
2,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0
3,854,SpotifyCares,False,Tue Oct 31 19:36:16 +0000 2017,"@115887 Hey! What device, operating system, an...",853,855.0
4,856,SpotifyCares,False,Tue Oct 31 22:25:16 +0000 2017,@115889 Got it. It's not possible at the momen...,857,858.0


In [59]:
import os
import json
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client ready!")

Gemini client ready!


In [33]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (84850, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Missing values:
tweet_id                       0
author_id                      0
inbound                        0
created_at                     0
text                           0
response_tweet_id          29479
in_response_to_tweet_id    26090
dtype: int64


In [34]:
# Support-agent messages are outbound (inbound = False)
support_accounts = df[df["inbound"] == False]["author_id"].value_counts()

print("Number of support accounts:", len(support_accounts))

print("\nTop 20 support accounts by number of tweets:")
print(support_accounts.head(20))

Number of support accounts: 1

Top 20 support accounts by number of tweets:
author_id
SpotifyCares    43265
Name: count, dtype: int64


In [35]:
# Map each tweet_id to its author_id
tweet_author = df.set_index("tweet_id")["author_id"].to_dict()

# For inbound customer messages, find which account they replied to
inbound_messages = df[df["inbound"] == True].copy()

inbound_messages["replied_to_account"] = inbound_messages[
    "in_response_to_tweet_id"
].map(tweet_author)

# Count customer messages directed to each support account
customer_volume = inbound_messages[
    inbound_messages["replied_to_account"].isin(support_accounts.index)
]["replied_to_account"].value_counts()

print("Top 20 brands by customer message volume:")
print(customer_volume.head(20))

Top 20 brands by customer message volume:
replied_to_account
SpotifyCares    11501
Name: count, dtype: int64


In [36]:
brands_to_check = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "VirginTrains"
]

for brand in brands_to_check:
    print("\n" + "=" * 80)
    print("BRAND:", brand)
    print("=" * 80)
    
    sample = inbound_messages[
        inbound_messages["replied_to_account"] == brand
    ].sample(5, random_state=42)
    
    for _, row in sample.iterrows():
        print("\nCustomer message:")
        print(row["text"])


BRAND: AmazonHelp


ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
brand_stats = []

for brand in customer_volume.head(20).index:
    brand_messages = inbound_messages[
        inbound_messages["replied_to_account"] == brand
    ]
    
    total = len(brand_messages)
    has_response = brand_messages["response_tweet_id"].notna().sum()
    response_rate = round((has_response / total) * 100, 2)
    
    brand_stats.append({
        "brand": brand,
        "customer_messages": total,
        "messages_with_response": has_response,
        "response_rate_%": response_rate
    })

brand_stats_df = pd.DataFrame(brand_stats)

print(
    brand_stats_df.sort_values(
        "messages_with_response",
        ascending=False
    ).head(15).to_string(index=False)
)

          brand  customer_messages  messages_with_response  response_rate_%
     AmazonHelp             100503                   69710            69.36
   AppleSupport              36658                   24907            67.94
   Uber_Support              22160                   12496            56.39
   SpotifyCares              15096                   12094            80.11
   VirginTrains              18450                   12020            65.15
    AmericanAir              18045                   10225            56.66
          Delta              14470                    9631            66.56
 VerizonSupport              12953                    8838            68.23
          Tesco              12812                    8575            66.93
        GWRHelp              13127                    8014            61.05
British_Airways              12485                    7649            61.27
    TMobileHelp              11662                    7227            61.97
    XboxSupp

In [37]:
spotify_messages = inbound_messages[
    inbound_messages["replied_to_account"] == "SpotifyCares"
].copy()

# Keep messages that have a support response
spotify_with_response = spotify_messages[
    spotify_messages["response_tweet_id"].notna()
].sample(10, random_state=42)

for _, row in spotify_with_response.iterrows():
    response_id = str(row["response_tweet_id"]).split(",")[0]
    response_id = int(float(response_id))
    
    response = df[df["tweet_id"] == response_id]["text"]
    
    print("\n" + "=" * 80)
    print("CUSTOMER:")
    print(row["text"])
    
    print("\nSPOTIFY SUPPORT:")
    if not response.empty:
        print(response.iloc[0])
    else:
        print("Response not found")


CUSTOMER:
@SpotifyCares Second (this is a usability thing): I was looking for a song of The National. 
Search box nowhere to be found.

SPOTIFY SUPPORT:
@436711 Got it. You can either use the web player (https://t.co/aT5o0wcN4V) or download the app at https://t.co/cgbgyQ4C7a for OS X and Windows /JP

CUSTOMER:
@SpotifyCares Thank you

SPOTIFY SUPPORT:
@205702 You're welcome! In case you have other concerns, don't hesitate to get back in touch. We're always... https://t.co/zOXI69QSqg /TM

CUSTOMER:
@SpotifyCares It's £15 a month a Spotify subscription (family account) and another £15 in data usage (at least!) a month. So essentially i'm paying £30 :(

SPOTIFY SUPPORT:
@283476 Could you DM us your account's username or email address? We'll have a look backstage and see what else we can suggest /AG https://t.co/ldFdZRiNAt

CUSTOMER:
@SpotifyCares It’s set to the United States.. I live in America

SPOTIFY SUPPORT:
@407204 Fingers crossed we'll be able to have it in the US soon. We'll let 

In [ ]:
# Show meaningful Spotify customer issues
meaningful_spotify = spotify_messages[
    spotify_messages["text"].str.len() > 40
].sample(20, random_state=42)

for i, (_, row) in enumerate(meaningful_spotify.iterrows(), 1):
    print(f"\n{i}. {row['text']}")


1. @SpotifyCares The adresses are: __email__ &amp; __email__

2. @SpotifyCares ok, thanks. he said let’s start a petition and I said to myself, no that’s not how you go about it and that’s when I tweeted you. I’ll pass along the info to him. Thanks!

3. @SpotifyCares There have been forums since like 2014

4. @SpotifyCares It’s not just my phone, my husbands 6s plus is the same way.

5. @SpotifyCares Voted for it, but I cannot stress enough how bad of a decision it is to ignore 2FA at this point in time. I hope they change their minds. https://t.co/IoN0G3ey7r

6. @SpotifyCares Hi. I have Android. I can't play a song. the name is "nacimos pa morir- Ánuel aa"

7. @SpotifyCares Sure, it’s the 6S with iOS 11.0.3. It just started happening with the most recent app update.

8. @SpotifyCares @436417 Hi, it’s not letting me download, saying I’ll not be able to play offline???

9. @SpotifyCares Yeah I noticed lol thanks 👍

10. @SpotifyCares Done that! Video prob doesn’t make much sense but it 

In [ ]:
spotify_review = spotify_messages[
    spotify_messages["text"].str.len() > 20
].sample(100, random_state=42)[["tweet_id", "text"]].copy()

spotify_review.to_csv(
    "../data/processed/spotify_review_sample.csv",
    index=False
)

print(spotify_review.head(20).to_string(index=False))

 tweet_id                                                                                                                                                                                                     text
   840516                                                                        @SpotifyCares Still an issue on both desktop + mobile. Says "this song is not available"...Most other music streams successfully.
  2582060                                                                                                                                                               @SpotifyCares I think I’m in love with you
   301478                                               @SpotifyCares Using it on the web.Playlists only show list of songs without checkmarks for saved songs. Also a radio tab no longer shows previous sessions
  1280144                                                                                                                          @SpotifyCares That page i

In [ ]:
# Create a larger sample of Spotify customer messages for inspection
spotify_check = spotify_messages[
    spotify_messages["text"].notna()
].sample(50, random_state=10)[["tweet_id", "text"]]

for i, (_, row) in enumerate(spotify_check.iterrows(), 1):
    print(f"{i}. {row['text']}\n")

1. @SpotifyCares Actually, I can't vote. it's "Case Closed" Really annoying. I would welcome any ideas to make the process easier.

2. @SpotifyCares 5 minutes anymore. Thanks!

3. @SpotifyCares The songs in my library.

4. @SpotifyCares Great, thanks: IPhone 6+ , IOS10.3.3 , Spotify version 8.4.22.515

5. @SpotifyCares Just closing and reopening the app fixes it, but it’s intermittent

6. @SpotifyCares I'm using Asus Zenfone 5, it's Android 4.3 (?) and my current Spotify version is 8.4.26.770 x86. Id the problem where I choose to storage the downloads? But I can't seem to change it (where I want to storage it).

7. @SpotifyCares Thanks 🙏🏻

8. @SpotifyCares Unfortunately all.

9. @SpotifyCares Hello. 😀 Thanks, but I'm on your waiting list for 2 years already 😂

10. @SpotifyCares Spotify Premium desktop app, Windows 10, 1.0.67.582.g19436fa3

11. @SpotifyCares Didn’t work!

12. @SpotifyCares  https://t.co/cI4mqeR6fZ

13. @SpotifyCares Yes it"s there, nr 3. How come it doesn't show on my a

In [38]:
# Show 30 Spotify customer messages that received a support reply
spotify_pairs = spotify_messages[
    spotify_messages["response_tweet_id"].notna()
].copy()

# Keep only messages with meaningful text
spotify_pairs = spotify_pairs[
    spotify_pairs["text"].str.len() >= 30
].sample(30, random_state=20)

for i, (_, row) in enumerate(spotify_pairs.iterrows(), 1):
    print(f"\n{'='*70}")
    print(f"{i}. CUSTOMER:")
    print(row["text"])


1. CUSTOMER:
@SpotifyCares 💚 https://t.co/VPKy97oO0L

2. CUSTOMER:
@SpotifyCares it finally showed up again, here ya go. https://t.co/nEBC58Mvns

3. CUSTOMER:
@SpotifyCares What do you mean pricing varies by payment mode? Can I avail the 9 pesos for 3 months using BDO Debit Card or Gcash Mastercard?

4. CUSTOMER:
@SpotifyCares Great! Thanks a lot! (:

5. CUSTOMER:
@SpotifyCares Spotify think I get wrong password, so I try to reset my password with myusername: linalthf, but I forgot what e-mail I used. Can u help me?

6. CUSTOMER:
@SpotifyCares El oh el so based on the fact of if I️ listen to them via your app vs purchasing their album like i did because I️ like to own my music. 👍🏾

7. CUSTOMER:
@SpotifyCares No worries , all sorted now

8. CUSTOMER:
@SpotifyCares It’s actually night time not day time ✌🏼

9. CUSTOMER:
@SpotifyCares omg thank you so much! That did help thxs

10. CUSTOMER:
@SpotifyCares i tried it. no solution

11. CUSTOMER:
@SpotifyCares Yeah the playlists stayed downlo

In [ ]:
apple_messages = inbound_messages[
    inbound_messages["replied_to_account"] == "AppleSupport"
].copy()

apple_pairs = apple_messages[
    apple_messages["response_tweet_id"].notna()
].copy()

apple_pairs = apple_pairs[
    apple_pairs["text"].str.len() >= 30
].sample(30, random_state=20)

for i, (_, row) in enumerate(apple_pairs.iterrows(), 1):
    print(f"\n{'='*70}")
    print(f"{i}. CUSTOMER:")
    print(row["text"])


1. CUSTOMER:
@AppleSupport It keeps switching to Denver time. Unclear why. It'll switch to Denver, then the Cupertino and then back. It's usually a day or 2 between.

2. CUSTOMER:
@AppleSupport I did that and it still leaves the question mark and the A

3. CUSTOMER:
@AppleSupport 11.0.3. I see there’s an update available, let’s see what happens

4. CUSTOMER:
@AppleSupport Hi! I followed all the step and nothing changed until now.

5. CUSTOMER:
@AppleSupport I’m seeing it on and off WiFi. It started a few days ago but it seems to be getting worse

6. CUSTOMER:
@AppleSupport Yes. Also happened before I upgraded OS.

7. CUSTOMER:
@AppleSupport Is not a buying question, IS about if Have to do something, some request if want to change the carrier in that version

8. CUSTOMER:
@AppleSupport You have no idea how mad I am that copy and paste was the answer all along. Nevertheless, thank you!

9. CUSTOMER:
@AppleSupport I click on download and install it asks for my password I put it in then i

In [ ]:
uber_messages = inbound_messages[
    inbound_messages["replied_to_account"] == "Uber_Support"
].copy()

uber_pairs = uber_messages[
    uber_messages["response_tweet_id"].notna()
].copy()

uber_pairs = uber_pairs[
    uber_pairs["text"].str.len() >= 30
].sample(30, random_state=20)

for i, (_, row) in enumerate(uber_pairs.iterrows(), 1):
    print(f"\n{'='*70}")
    print(f"{i}. CUSTOMER:")
    print(row["text"])


1. CUSTOMER:
@Uber_Support Did that last night. We did want the food originally but the restaurant closed just as the food came plus I wouldn’t want to spend money again on the food and delivery 😒😒

2. CUSTOMER:
@Uber_Support Done. Please help

3. CUSTOMER:
@Uber_Support Emailed yesterday. No response?

4. CUSTOMER:
@Uber_Support @146917 we don't have a no to call in Uber for support while drivers are messing up passengers time. Why Uber is allowed in India

5. CUSTOMER:
@Uber_Support They've responded but not about this specific issue with voiceover freezing when cancelling a pool trip.

6. CUSTOMER:
@Uber_Support I would like to know if my area, although while not “Miami” would be considered to be a part of the Miami Pass Area. I received the offer but I want to make sure I’ll be able to use it if I opt-in to it

7. CUSTOMER:
@Uber_Support Visited during opening times   told by guard  closed till Mon. It's an hour from home. Is there a contact number? @131434 @131077

8. CUSTOMER:
@

In [ ]:
# Start with all customer messages directed to Spotify
spotify_clean = inbound_messages[
    inbound_messages["replied_to_account"] == "SpotifyCares"
].copy()

# Keep meaningful messages
spotify_clean = spotify_clean[
    spotify_clean["text"].notna() &
    (spotify_clean["text"].str.len() >= 20)
].copy()

print("Total Spotify customer messages:", len(spotify_clean))

spotify_clean[["tweet_id", "text", "response_tweet_id"]].head(10)

Total Spotify customer messages: 14717


,tweet_id,text,response_tweet_id
541,849,@SpotifyCares doesn’t work and i even tried de...,851
543,850,@SpotifyCares Premium &amp; when i️ have it on...,848
545,853,@SpotifyCares iphone 7+ and i have the most re...,852
549,857,"@SpotifyCares Yes, multiple times. No changes....",859
551,858,@SpotifyCares 2/2... and there is no way to ma...,856
555,864,@SpotifyCares ok thx,863
1272,1868,@SpotifyCares I tried it on web browser and it...,1870
1275,1872,@SpotifyCares 1.0.65.320.gac7a8e02,1873
1277,1869,@SpotifyCares using a MacBook Pro with OS X El...,1867
1281,1877,"@SpotifyCares Made In Germany, Du Bist Gut, Co...","1879,1880"


In [ ]:
# Remove duplicate messages
spotify_clean = spotify_clean.drop_duplicates(subset=["text"]).copy()

# Remove very short / low-information messages
spotify_clean = spotify_clean[
    spotify_clean["text"].str.len() >= 30
].copy()

print("Messages after removing duplicates and short texts:", len(spotify_clean))

Messages after removing duplicates and short texts: 13219


In [ ]:
# Check customer messages that have a linked Spotify support response

with_response = spotify_clean[
    spotify_clean["response_tweet_id"].notna()
].copy()

without_response = spotify_clean[
    spotify_clean["response_tweet_id"].isna()
].copy()

print("Messages WITH Spotify response:", len(with_response))
print("Messages WITHOUT Spotify response:", len(without_response))

Messages WITH Spotify response: 10906
Messages WITHOUT Spotify response: 2313


In [ ]:
# Take 5 customer messages with responses
sample_pairs = with_response.sample(5, random_state=42)

for _, row in sample_pairs.iterrows():
    response_id = str(row["response_tweet_id"]).split(",")[0]
    
    response = df[
        df["tweet_id"].astype(str) == response_id
    ]
    
    print("\n" + "=" * 80)
    print("CUSTOMER:")
    print(row["text"])
    
    print("\nSPOTIFY RESPONSE:")
    if not response.empty:
        print(response.iloc[0]["text"])
    else:
        print("Response not found")


CUSTOMER:
@SpotifyCares Hi! Sorry for not getting back to you. I’ve got premium Spotify, on MacBook Air macOS Sierra version 10.12.6

SPOTIFY RESPONSE:
@396540 Got it, thanks for that info! Are you having issues playing tracks on any other devices? Let us know /RV

CUSTOMER:
@SpotifyCares Thanks for seeing this! Hopefully one day this may be a promotion because I’d be signing up ASAP!

SPOTIFY RESPONSE:
@547412 No worries! Give us a shout if you have any other questions 🙂 /JL

CUSTOMER:
@SpotifyCares This is the latest version of Spotify on Windows 10 Mobile. https://t.co/v0ZZMdAmNq

SPOTIFY RESPONSE:
@169835 Got it. The Spotify app on Windows Phone is currently in maintenance mode. There's some more info here: https://t.co/KSq4qsvF5c /PB

CUSTOMER:
@SpotifyCares Right at the beginning and the second chorus, next track did it too, not sure of the rest as I turned it off

SPOTIFY RESPONSE:
@555656 The tracks are playing fine for us. Does the issue persist for you on a different device?

In [ ]:
# Create a lookup of all tweets by tweet_id
tweet_lookup = df.set_index(
    df["tweet_id"].astype(str)
)["text"].to_dict()

# Create clean customer-support pairs
pairs = []

for _, row in with_response.iterrows():
    response_id = str(row["response_tweet_id"]).split(",")[0].strip()
    
    support_response = tweet_lookup.get(response_id)
    
    if support_response:
        pairs.append({
            "customer_tweet_id": row["tweet_id"],
            "customer_message": row["text"],
            "support_response": support_response
        })

pairs_df = pd.DataFrame(pairs)

print("Total valid customer-support pairs:", len(pairs_df))

pairs_df.head()

Total valid customer-support pairs: 10906


,customer_tweet_id,customer_message,support_response
0,849,@SpotifyCares doesn’t work and i even tried de...,@115887 Could you send us a DM with your accou...
1,850,@SpotifyCares Premium &amp; when i️ have it on...,@115887 Hmm. Can you try restarting your devic...
2,853,@SpotifyCares iphone 7+ and i have the most re...,"@115887 Thanks. Just to be sure, are you Free ..."
3,857,"@SpotifyCares Yes, multiple times. No changes....",@115889 Sorry to hear that. The Spotify app on...
4,858,@SpotifyCares 2/2... and there is no way to ma...,@115889 Got it. It's not possible at the momen...


In [ ]:
sample_check = pairs_df.sample(10, random_state=10)

for i, (_, row) in enumerate(sample_check.iterrows(), 1):
    print("\n" + "=" * 100)
    print(f"PAIR {i}")
    print("-" * 100)
    print("CUSTOMER:")
    print(row["customer_message"])
    print("\nSPOTIFY SUPPORT:")
    print(row["support_response"])


PAIR 1
----------------------------------------------------------------------------------------------------
CUSTOMER:
@SpotifyCares It's for my son so let me get his info

SPOTIFY SUPPORT:
@653044 Sure! We'll be right here waiting /KM https://t.co/ldFdZRiNAt

PAIR 2
----------------------------------------------------------------------------------------------------
CUSTOMER:
@SpotifyCares Thanks. Was hoping you might be helpful and confirm whether it is known/being addressed.

SPOTIFY SUPPORT:
@678418 Sorry to hear you feel that way. We'd be happy to keep helping out. Does logging out, restarting your phone, and logging back in help? /JE

PAIR 3
----------------------------------------------------------------------------------------------------
CUSTOMER:
@SpotifyCares https://t.co/Vk2PHrZHXS &lt;== this is secrets official. No idea wth this is ==&gt; https://t.co/bpNThePc3S

SPOTIFY SUPPORT:
@369362 Thanks. We'll get this reported. Great detective work! https://t.co/LpsmyUtme0 /DR

PA

In [ ]:
# Check whether customer messages are original messages
# or replies to an earlier tweet

root_messages = with_response[
    with_response["in_response_to_tweet_id"].isna()
].copy()

reply_messages = with_response[
    with_response["in_response_to_tweet_id"].notna()
].copy()

print("Original customer messages:", len(root_messages))
print("Customer follow-up/reply messages:", len(reply_messages))

Original customer messages: 0
Customer follow-up/reply messages: 10906


In [ ]:
print(df.columns.tolist())

print("\n--- Sample Spotify customer messages ---")

print(
    spotify_clean[
        ["tweet_id", "in_response_to_tweet_id", "response_tweet_id", "text"]
    ].head(10).to_string(index=False)
)

['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

--- Sample Spotify customer messages ---
 tweet_id  in_response_to_tweet_id response_tweet_id                                                                                                                                                                  text
      849                    848.0               851                                                                                                          @SpotifyCares doesn’t work and i even tried deleting the app
      850                    852.0               848         @SpotifyCares Premium &amp; when i️ have it on shuffle it turns off when the song is done and just plays in order and the repeat lights up but doesn’t repeat
      853                    854.0               852                                                                                                 @SpotifyCares iphone 7+ and i have the most r

In [ ]:
# Create a lookup for tweets
df_lookup = df.set_index(df["tweet_id"].astype(str)).to_dict("index")

# Start with Spotify customer tweet 849
current_id = "849"

conversation = []

while current_id in df_lookup:
    tweet = df_lookup[current_id]
    
    conversation.append({
        "tweet_id": current_id,
        "author": tweet["author_id"],
        "inbound": tweet["inbound"],
        "text": tweet["text"]
    })
    
    parent_id = tweet["in_response_to_tweet_id"]
    
    if pd.isna(parent_id):
        break
        
    current_id = str(int(parent_id))

# Reverse so original message appears first
conversation.reverse()

for msg in conversation:
    print("\n" + "=" * 80)
    print("TWEET ID:", msg["tweet_id"])
    print("AUTHOR:", msg["author"])
    print("INBOUND:", msg["inbound"])
    print("MESSAGE:", msg["text"])

In [ ]:
# Find Spotify customer messages that START a conversation
# They are inbound messages with no previous tweet

spotify_original = df[
    (df["inbound"] == True) &
    (df["in_response_to_tweet_id"].isna()) &
    (df["response_tweet_id"].notna())
].copy()

# Keep only conversations where SpotifyCares is the next responder
spotify_original = spotify_original[
    spotify_original["response_tweet_id"].astype(str).isin(
        df[df["author_id"] == "SpotifyCares"]["tweet_id"].astype(str)
    )
].copy()

# Keep meaningful messages
spotify_original = spotify_original[
    spotify_original["text"].notna() &
    (spotify_original["text"].str.len() >= 30)
].copy()

print("Original Spotify customer issues:", len(spotify_original))

print(
    spotify_original[
        ["tweet_id", "text", "response_tweet_id"]
    ].head(10).to_string(index=False)
)

Original Spotify customer issues: 24193
 tweet_id                                                                                                                                                                                                                                                         text response_tweet_id
      855                                                                                                                                                           i’m pissed my @115888 shuffle and repeat button just don’t fucking work and i’m getting frustrated               854
      862                                                                                                                            @SpotifyCares @115890 Groove Music quits &amp; redirect to Spotify. But the W10M App is a bugfest.. Any further Updates possible?               860
      866                                                                                                            

In [ ]:

sample_original = spotify_original.sample(20, random_state=42)

for i, (_, row) in enumerate(sample_original.iterrows(), 1):
    print(f"\n{i}. {row['text']}")


1. Why it’s not on @115888 the new @118062 album #reputation? 👎🏻

2. @SpotifyCares I pay for premium and I’m fed up. doesn’t work!! I want this sorted or I’m cancelling my subscription.

3. My @115888 Discover Weekly playlist has gone somewhere I cannot follow. Any way to “reset” it? Sometimes data science just doesn’t work.

4. It's really frustrating that @115888​ is still not available in India where a lot of my readers are based. https://t.co/ZHRgEWa4kc

5. @SpotifyCares I can't seem to delete a playlist, and it's not available either? But it won't go from my playlists bar...

6. Hi, I'm trying to pay for my Spotify premium subscription with PayPal and it's not going through? @115888

7. Spotify is so confused with my odd music tastes. It tries so hard to give new suggestions though.

8. @115888 is it possible to change “behind the lyrics” to just “lyrics”? Am I being stupid or am I unable to change it?

9. @SpotifyCares when will you update the app to support the iPhone X ?

10. 

In [ ]:
import re
import html

def clean_text(text):
    # Convert HTML entities like &amp;
    text = html.unescape(text)
    
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    
    # Remove Twitter IDs like @115888
    text = re.sub(r'@\d+', '', text)
    
    # Remove Spotify mention
    text = re.sub(r'@SpotifyCares', '', text, flags=re.IGNORECASE)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

spotify_original["clean_text"] = spotify_original["text"].apply(clean_text)

print("Before:")
print(spotify_original["text"].iloc[0])

print("\nAfter:")
print(spotify_original["clean_text"].iloc[0])

Before:
i’m pissed my @115888 shuffle and repeat button just don’t fucking work and i’m getting frustrated

After:
i’m pissed my shuffle and repeat button just don’t fucking work and i’m getting frustrated


In [ ]:
# Remove empty cleaned messages
spotify_original = spotify_original[
    spotify_original["clean_text"].str.len() >= 20
].copy()

# Remove duplicate cleaned messages
spotify_original = spotify_original.drop_duplicates(
    subset=["clean_text"]
).copy()

print("Final cleaned Spotify issues:", len(spotify_original))
print("Empty values:", spotify_original["clean_text"].isna().sum())

Final cleaned Spotify issues: 23748
Empty values: 0


In [ ]:
print("spotify_original shape:", spotify_original.shape)

print("\nFirst 5 rows:")
print(
    spotify_original[
        ["tweet_id", "author_id", "inbound", "in_response_to_tweet_id", "clean_text"]
    ].head().to_string(index=False)
)

spotify_original shape: (23748, 8)

First 5 rows:
 tweet_id author_id  inbound  in_response_to_tweet_id                                                                                                                        clean_text
      855    115887     True                      NaN                                        i’m pissed my shuffle and repeat button just don’t fucking work and i’m getting frustrated
      862    115889     True                      NaN                           Groove Music quits & redirect to Spotify. But the W10M App is a bugfest.. Any further Updates possible?
      866    115891     True                      NaN                                                                       is there a way to find non-explicit songs that are explicit
     1875    116128     True                      NaN                                         Why are my devices not doing the cool thing when I play the #StrangerThings albums on ? 😭
     1878    116129     True  

In [ ]:
# Rebuild Spotify original customer issues from scratch

# Get all SpotifyCares tweet IDs
spotify_support_ids = set(
    df.loc[
        df["author_id"] == "SpotifyCares",
        "tweet_id"
    ].astype(str)
)

# Find original inbound customer messages
spotify_original = df[
    (df["inbound"] == True) &
    (df["in_response_to_tweet_id"].isna()) &
    (df["response_tweet_id"].notna())
].copy()

# Keep only messages whose next response is from SpotifyCares
spotify_original = spotify_original[
    spotify_original["response_tweet_id"]
    .astype(str)
    .str.split(",")
    .apply(lambda ids: any(tweet_id.strip() in spotify_support_ids for tweet_id in ids))
].copy()

print("Spotify original customer issues:", len(spotify_original))

print(
    spotify_original[
        ["tweet_id", "author_id", "response_tweet_id", "text"]
    ].head(10).to_string(index=False)
)

Spotify original customer issues: 26068
 tweet_id author_id response_tweet_id                                                                                                                                                                                                                                                                                  text
      855    115887               854                                                                                                                                                                                    i’m pissed my @115888 shuffle and repeat button just don’t fucking work and i’m getting frustrated
      862    115889               860                                                                                                                                                     @SpotifyCares @115890 Groove Music quits &amp; redirect to Spotify. But the W10M App is a bugfest.. Any further Updates possible?
      866   

In [ ]:
# Clean the customer messages
spotify_original["clean_text"] = spotify_original["text"].apply(clean_text)

# Remove empty/very short messages
spotify_clean = spotify_original[
    spotify_original["clean_text"].str.len() >= 20
].copy()

# Remove duplicate customer issues
spotify_clean = spotify_clean.drop_duplicates(
    subset=["clean_text"]
).copy()

print("Final cleaned Spotify customer issues:", len(spotify_clean))
print("Empty values:", spotify_clean["clean_text"].isna().sum())

spotify_clean[
    ["tweet_id", "author_id", "clean_text", "response_tweet_id"]
].head(10)

Final cleaned Spotify customer issues: 25290
Empty values: 0


,tweet_id,author_id,clean_text,response_tweet_id
547,855,115887,i’m pissed my shuffle and repeat button just d...,854
553,862,115889,Groove Music quits & redirect to Spotify. But ...,860
557,866,115891,is there a way to find non-explicit songs that...,865
1279,1875,116128,Why are my devices not doing the cool thing wh...,1874
1284,1878,116129,why are Nena's releases between 2002 and 2012 ...,1876
2061,2804,116375,So : three days later you’re still advertising...,2803
2064,2807,116376,- you having a party without me?,2805
2066,2809,116377,. any way to block this particular ad? Happy w...,2808
2073,2855,116379,"I've been trying to select ""no"" for this optio...",2854
2076,2858,116381,"Hey guys, I tried to update my payment method ...","2857,2859"


In [ ]:
print("Total customer issues:", len(spotify_clean))
print("Unique customer messages:", spotify_clean["clean_text"].nunique())
print("Messages with Spotify response:", spotify_clean["response_tweet_id"].notna().sum())
print("Unique support response IDs:", spotify_clean["response_tweet_id"].nunique())

Total customer issues: 25290
Unique customer messages: 25290
Messages with Spotify response: 25290
Unique support response IDs: 25290


In [ ]:
# Get the actual Spotify support response text

spotify_responses = df[
    df["tweet_id"].astype(str).isin(
        spotify_clean["response_tweet_id"].astype(str)
    )
][["tweet_id", "text"]].copy()

spotify_responses.columns = [
    "response_tweet_id",
    "support_response"
]

# Convert IDs to same type before merging
spotify_clean["response_tweet_id"] = spotify_clean[
    "response_tweet_id"
].astype(str)

spotify_responses["response_tweet_id"] = spotify_responses[
    "response_tweet_id"
].astype(str)

# Merge customer issues with Spotify responses
spotify_pairs = spotify_clean.merge(
    spotify_responses,
    on="response_tweet_id",
    how="inner"
)

print("Final customer-support pairs:", len(spotify_pairs))

spotify_pairs[
    ["clean_text", "support_response"]
].head(10)

Final customer-support pairs: 23823


,clean_text,support_response
0,i’m pissed my shuffle and repeat button just d...,"@115887 Hey! What device, operating system, an..."
1,Groove Music quits & redirect to Spotify. But ...,@115889 Hey there! That doesn't sound good. Wh...
2,is there a way to find non-explicit songs that...,@115891 Hey Mikey! We're afraid there's no way...
3,Why are my devices not doing the cool thing wh...,@116128 Hey Jacklynn! The cavalry's here. Coul...
4,why are Nena's releases between 2002 and 2012 ...,@116129 Hi Liam! Could you let us know the tra...
5,So : three days later you’re still advertising...,"@116375 Hey there! Just to clarify, which part..."
6,- you having a party without me?,"@116376 Hey Tom! Not to worry, we're more than..."
7,. any way to block this particular ad? Happy w...,@116377 Hey there! Thanks for taking the time ...
8,"I've been trying to select ""no"" for this optio...",@116379 Hey Dylan! Can you let us know which v...
9,why is this album on Spotify if it's not playa...,@116384 Hey there! Sometimes content gets temp...


In [ ]:
print("Total pairs:", len(spotify_pairs))
print("Empty customer messages:", spotify_pairs["clean_text"].isna().sum())
print("Empty support responses:", spotify_pairs["support_response"].isna().sum())
print("Duplicate customer messages:", spotify_pairs["clean_text"].duplicated().sum())
print("Duplicate support responses:", spotify_pairs["support_response"].duplicated().sum())

Total pairs: 23823
Empty customer messages: 0
Empty support responses: 0
Duplicate customer messages: 0
Duplicate support responses: 1


In [ ]:
# Remove any duplicate customer-support pairs
spotify_pairs = spotify_pairs.drop_duplicates(
    subset=["clean_text", "support_response"]
).copy()

print("Final unique customer-support pairs:", len(spotify_pairs))

Final unique customer-support pairs: 23823


In [ ]:
spotify_pairs[
    ["clean_text", "support_response"]
].to_csv(
    "spotify_final_pairs.csv",
    index=False
)

print("Dataset saved successfully!")
print("Final shape:", spotify_pairs.shape)

Dataset saved successfully!
Final shape: (23823, 9)


In [42]:
print(spotify_pairs.columns.tolist())
print(spotify_pairs.head())

['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'replied_to_account']
       tweet_id author_id  inbound                      created_at  \
48275    466048    225882     True  Fri Dec 01 05:29:18 +0000 2017   
47676    409783    209251     True  Thu Oct 19 02:21:38 +0000 2017   
46811    327640    194110     True  Sat Nov 25 04:16:19 +0000 2017   
44454    116585    141975     True  Thu Nov 23 21:33:01 +0000 2017   
43573     39773    124711     True  Wed Nov 01 15:06:59 +0000 2017   

                                                    text response_tweet_id  \
48275            @SpotifyCares 💚 https://t.co/VPKy97oO0L            466049   
47676  @SpotifyCares it finally showed up again, here...            409785   
46811  @SpotifyCares What do you mean pricing varies ...            327637   
44454              @SpotifyCares Great! Thanks a lot! (:            116584   
43573  @SpotifyCares Spotify think I get wrong passwo...    

In [43]:
print(spotify_pairs[[
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id",
    "inbound",
    "replied_to_account"
]].head(10).to_string())

                                                                                                                                                             text response_tweet_id  in_response_to_tweet_id  inbound replied_to_account
48275                                                                                                                     @SpotifyCares 💚 https://t.co/VPKy97oO0L            466049                 466047.0     True       SpotifyCares
47676                                                                               @SpotifyCares it finally showed up again, here ya go. https://t.co/nEBC58Mvns            409785                 409782.0     True       SpotifyCares
46811               @SpotifyCares What do you mean pricing varies by payment mode? Can I avail the 9 pesos for 3 months using BDO Debit Card or Gcash Mastercard?            327637                 327642.0     True       SpotifyCares
44454                                                               

In [46]:
import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print("\nDATAFRAME:", name)
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())


DATAFRAME: __
Shape: (5, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DATAFRAME: ___
Shape: (5, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DATAFRAME: df
Shape: (84850, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DATAFRAME: _1
Shape: (5, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DATAFRAME: inbound_messages
Shape: (41585, 8)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'replied_to_account']

DATAFRAME: _7
Shape: (5, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

DATAFRAME: _13
Shape: (5, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_

In [44]:
print("Inbound counts:")
print(spotify_pairs["inbound"].value_counts())

Inbound counts:
inbound
True    30
Name: count, dtype: int64


In [47]:
import pandas as pd

# Use the original full dataset
tweets = df.copy()

# Convert IDs to strings safely
tweets["tweet_id"] = tweets["tweet_id"].astype(str).str.strip()

# Get customer messages that have a response
customers = tweets[
    (tweets["inbound"] == True) &
    (tweets["response_tweet_id"].notna())
].copy()

# response_tweet_id can sometimes contain multiple IDs
customers["response_tweet_id"] = customers["response_tweet_id"].astype(str)

customers["response_id"] = customers["response_tweet_id"].str.split(",")

customers = customers.explode("response_id")

customers["response_id"] = customers["response_id"].str.strip()

# Get all actual reply tweets
replies = tweets[
    tweets["inbound"] == False
][["tweet_id", "text"]].copy()

replies.columns = ["response_id", "support_response"]

# Merge customer messages with ACTUAL support reply text
paired_data = customers.merge(
    replies,
    on="response_id",
    how="inner"
)

# Keep only required columns
ai_dataset = paired_data[
    ["text", "support_response"]
].copy()

ai_dataset.columns = [
    "customer_issue",
    "support_response"
]

# Remove empty/null values
ai_dataset = ai_dataset.dropna()

ai_dataset = ai_dataset[
    (ai_dataset["customer_issue"].str.strip() != "") &
    (ai_dataset["support_response"].str.strip() != "")
].reset_index(drop=True)

print("AI Dataset created successfully!")
print("Shape:", ai_dataset.shape)

print("\n--- SAMPLE PAIRS ---")
print(ai_dataset.head(10).to_string())

AI Dataset created successfully!
Shape: (43092, 2)

--- SAMPLE PAIRS ---
                                                                                                                                                  customer_issue                                                                                                                            support_response
0                                                                                                   @SpotifyCares doesn’t work and i even tried deleting the app                   @115887 Could you send us a DM with your account's email address? We'll take a look backstage /CH https://t.co/ldFdZRiNAt
1  @SpotifyCares Premium &amp; when i️ have it on shuffle it turns off when the song is done and just plays in order and the repeat lights up but doesn’t repeat          @115887 Hmm. Can you try restarting your device by holding the Sleep/Wake + Volume Down buttons for 10 seconds? Keep us posted /LS
2                       

In [48]:
print(ai_dataset.shape)
print(ai_dataset.head(3).to_string())

(43092, 2)
                                                                                                                                                  customer_issue                                                                                                                      support_response
0                                                                                                   @SpotifyCares doesn’t work and i even tried deleting the app             @115887 Could you send us a DM with your account's email address? We'll take a look backstage /CH https://t.co/ldFdZRiNAt
1  @SpotifyCares Premium &amp; when i️ have it on shuffle it turns off when the song is done and just plays in order and the repeat lights up but doesn’t repeat    @115887 Hmm. Can you try restarting your device by holding the Sleep/Wake + Volume Down buttons for 10 seconds? Keep us posted /LS
2                                                                                          @SpotifyCares

In [49]:
# Create reproducible golden evaluation set

golden_set = ai_dataset.sample(
    n=200,
    random_state=42
).copy()

golden_set = golden_set.reset_index(drop=True)

golden_set.insert(
    0,
    "golden_id",
    range(1, len(golden_set) + 1)
)

# Human evaluation columns
golden_set["human_relevance"] = ""
golden_set["human_helpfulness"] = ""
golden_set["human_response_quality"] = ""
golden_set["human_notes"] = ""

print("Golden evaluation set created successfully!")
print("Shape:", golden_set.shape)

golden_set.head()

Golden evaluation set created successfully!
Shape: (200, 7)


,golden_id,customer_issue,support_response,human_relevance,human_helpfulness,human_response_quality,human_notes
0,1,@SpotifyCares please help me cancel my payment...,@486668 Hey Brandon! We've already sent a resp...,,,,
1,2,@SpotifyCares for DVSN plsss and thank you,@624384 Thanks. Pre-sale codes are being sent ...,,,,
2,3,"@SpotifyCares Hey, I sent a DM. Please check. ...",@503793 Hey there! We've just sent you a DM 🙂 /DF,,,,
3,4,@SpotifyCares it’s been over 2 months 🤧,"@756275 Hey Joanna, we hear you! Sometimes con...",,,,
4,5,@SpotifyCares And I love the feature on my oth...,"@123421 You're welcome! For anything else, jus...",,,,


In [50]:
print(ai_dataset.shape)

print("\nCustomer Issue:")
print(ai_dataset.iloc[0]["customer_issue"])

print("\nActual Support Response:")
print(ai_dataset.iloc[0]["support_response"])

(43092, 2)

Customer Issue:
@SpotifyCares doesn’t work and i even tried deleting the app

Actual Support Response:
@115887 Could you send us a DM with your account's email address? We'll take a look backstage /CH https://t.co/ldFdZRiNAt


In [41]:
# Create final dataset for AI Ticket System

ai_dataset = spotify_pairs[
    ["clean_text", "support_response"]
].copy()

ai_dataset.columns = [
    "customer_issue",
    "support_response"
]

print("AI Dataset shape:", ai_dataset.shape)

ai_dataset.head(10)

KeyError: "None of [Index(['clean_text', 'support_response'], dtype='object')] are in the [columns]"

In [ ]:
ai_dataset.to_csv(
    "spotify_ai_dataset.csv",
    index=False
)

print("AI-ready dataset saved!")

AI-ready dataset saved!


In [ ]:
# Clean support responses

ai_dataset["support_response"] = ai_dataset["support_response"].apply(clean_text)

# Remove empty or very short responses
ai_dataset = ai_dataset[
    ai_dataset["support_response"].str.len() >= 10
].copy()

# Remove duplicate issue-response pairs
ai_dataset = ai_dataset.drop_duplicates(
    subset=["customer_issue", "support_response"]
).copy()

print("Final AI dataset shape:", ai_dataset.shape)
print("Empty issues:", ai_dataset["customer_issue"].isna().sum())
print("Empty responses:", ai_dataset["support_response"].isna().sum())

ai_dataset.head(10)

Final AI dataset shape: (23815, 2)
Empty issues: 0
Empty responses: 0


,customer_issue,support_response
0,i’m pissed my shuffle and repeat button just d...,"Hey! What device, operating system, and Spotif..."
1,Groove Music quits & redirect to Spotify. But ...,Hey there! That doesn't sound good. What's hap...
2,is there a way to find non-explicit songs that...,Hey Mikey! We're afraid there's no way to filt...
3,Why are my devices not doing the cool thing wh...,Hey Jacklynn! The cavalry's here. Could you te...
4,why are Nena's releases between 2002 and 2012 ...,Hi Liam! Could you let us know the tracks/albu...
5,So : three days later you’re still advertising...,"Hey there! Just to clarify, which part of Cana..."
6,- you having a party without me?,"Hey Tom! Not to worry, we're more than happy t..."
7,. any way to block this particular ad? Happy w...,Hey there! Thanks for taking the time to reach...
8,"I've been trying to select ""no"" for this optio...",Hey Dylan! Can you let us know which versions ...
9,why is this album on Spotify if it's not playa...,Hey there! Sometimes content gets temporarily ...


In [ ]:
ai_dataset.to_csv(
    "spotify_ai_final_dataset.csv",
    index=False
)

print("Final AI dataset saved successfully!")
print("Shape:", ai_dataset.shape)

Final AI dataset saved successfully!
Shape: (23815, 2)


In [ ]:
!pip install sentence-transformers faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
from sentence_transformers import SentenceTransformer

# Load a lightweight model for semantic search
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 758.00it/s]


Embedding model loaded successfully!


In [52]:
issue_embeddings = model.encode(
    ai_dataset["customer_issue"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", issue_embeddings.shape)

Batches: 100%|██████████| 1347/1347 [10:02<00:00,  2.23it/s]


Embeddings shape: (43092, 384)


In [53]:
import faiss
import numpy as np

dimension = issue_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(issue_embeddings.astype("float32"))

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 43092


In [ ]:
query = "I cannot play songs on my Spotify app"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = index.search(query_embedding, k=5)

print("Top 5 similar customer issues:\n")

for i, idx in enumerate(indices[0], 1):
    print(f"{i}. {ai_dataset.iloc[idx]['customer_issue']}")
    print("Support response:", ai_dataset.iloc[idx]["support_response"])
    print()

Top 5 similar customer issues:

1. I'm having so many problems with the Spotify app on my phone with songs not playing
Support response: Hey! Can you let us know which device and operating system you're using? Also, is this happening over WiFi or 3G/4G? /NQ

2. why am I unable to play ANY songs on spotify? I've tried logging out and logging back in, nothing is helping
Support response: Hey Celia, that doesn't sound good. Can you let us know what device, operating system, and Spotify version you're using? /RK

3. Hey there, I'm having trouble with my spotify app playing music, all songs are "unavailable" restarting spotify didn't fix
Support response: Hey Kevin! Can you let us know the device/OS and Spotify version you're using? We'll see what we can suggest /AR

4. Hi, I can't play music with my Spotify desktop, can u help me ?
Support response: Hey Yulio, help's here! Just to check, are you trying to stream your music online or play your offline tracks? /JN

5. I have an issue with my

In [ ]:
def search_similar_issues(query, k=5):
    
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    
    distances, indices = index.search(query_embedding, k)
    
    results = []
    
    for idx in indices[0]:
        results.append({
            "customer_issue": ai_dataset.iloc[idx]["customer_issue"],
            "support_response": ai_dataset.iloc[idx]["support_response"]
        })
    
    return results

In [ ]:
results = search_similar_issues(
    "Spotify songs are not playing",
    k=3
)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Customer issue:", result["customer_issue"])
    print("Support response:", result["support_response"])


--- Result 1 ---
Customer issue: spotify isnt working
Support response: Hey! We were off beat for a second, but we’re back on track now (excuse the puns) /CE

--- Result 2 ---
Customer issue: I can't listen to specific artists on spotify, it shows them but music won't play
Support response: Hi! We've just sent you a bit more info over DM. We'll carry on helping out there /AR

--- Result 3 ---
Customer issue: Spotify is saying every song I try to play is "not available", possibly after an update. Any ideas?
Support response: Hey there! That's not cool. Does logging out > restarting the device > logging back in help? Keep us posted /PX


In [ ]:
# Save the FAISS index and dataset for later use

import pickle

faiss.write_index(index, "spotify_faiss_index.index")

with open("spotify_ai_dataset.pkl", "wb") as f:
    pickle.dump(ai_dataset, f)

print("FAISS index and dataset saved successfully!")

FAISS index and dataset saved successfully!


In [ ]:
# Create a function to retrieve relevant context for the AI

def get_rag_context(query, k=3):
    
    results = search_similar_issues(query, k)
    
    context = ""
    
    for i, result in enumerate(results, 1):
        context += f"""
Example {i}
Customer Issue: {result['customer_issue']}
Support Response: {result['support_response']}

"""
    
    return context

In [ ]:
# Test RAG context retrieval

query = "My Spotify music is not playing"

context = get_rag_context(query, k=3)

print(context)


Example 1
Customer Issue: spotify isnt working
Support Response: Hey! We were off beat for a second, but we’re back on track now (excuse the puns) /CE


Example 2
Customer Issue: my spotify isn’t working help
Support Response: Hey Aurora! Can you let us know what's happening exactly? We'll see what we can suggest /FR


Example 3
Customer Issue: Spotify isn't working and I don't know how to fix it
Support Response: Hey there, help's here! Can you DM us exactly what's happening? We'll see what we can suggest 🙂 /CG




In [ ]:
# Create the prompt template for the AI agent

def create_support_prompt(customer_query):
    
    context = get_rag_context(customer_query, k=3)
    
    prompt = f"""
You are a helpful customer support AI agent.

Use the historical customer issues and support responses below as reference.

{context}

Current Customer Issue:
{customer_query}

Generate a helpful, clear, and polite support response.
Do not copy the historical responses exactly.
"""
    
    return prompt

In [ ]:
# Test the final RAG prompt

customer_query = "Spotify stops playing music after a few songs"

final_prompt = create_support_prompt(customer_query)

print(final_prompt)


You are a helpful customer support AI agent.

Use the historical customer issues and support responses below as reference.


Example 1
Customer Issue: hey! My Spotify randomly stops playing music and it’s happening everytime I listen... :-(
Support Response: Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK


Example 2
Customer Issue: My Spotify keeps pausing and stops and then I go to it and can't play my music without exiting the app and coming back. Help.
Support Response: Hey! Can you let us know which device and operating system you're using? We'll see what we can suggest /NQ


Example 3
Customer Issue: aye whenever I leave the Spotify app my music stops playing. And sometimes i will have headphones in and it’ll just stop playing all together. Please help me ;(
Support Response: Hey! That doesn't sound good. Can you let us know which device/OS and Spotify version you're using? We'll see what we 

In [ ]:
# Create a basic RAG support agent using the best retrieved solution

def generate_support_response(customer_query):
    
    results = search_similar_issues(customer_query, k=3)
    
    best_result = results[0]
    
    return {
        "customer_query": customer_query,
        "similar_issue": best_result["customer_issue"],
        "suggested_response": best_result["support_response"]
    }

In [ ]:
# Test the RAG support agent

ticket = generate_support_response(
    "Spotify stops playing music after a few songs"
)

print("CUSTOMER QUERY:")
print(ticket["customer_query"])

print("\nMOST SIMILAR PAST ISSUE:")
print(ticket["similar_issue"])

print("\nSUGGESTED SUPPORT RESPONSE:")
print(ticket["suggested_response"])

CUSTOMER QUERY:
Spotify stops playing music after a few songs

MOST SIMILAR PAST ISSUE:
hey! My Spotify randomly stops playing music and it’s happening everytime I listen... :-(

SUGGESTED SUPPORT RESPONSE:
Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK


In [ ]:
def search_with_scores(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            "customer_issue": ai_dataset.iloc[idx]["customer_issue"],
            "support_response": ai_dataset.iloc[idx]["support_response"],
            "distance": float(distance)
        })

    return results

In [ ]:
results = search_with_scores(
    "Spotify stops playing music after a few songs",
    k=3
)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Issue:", result["customer_issue"])
    print("Response:", result["support_response"])
    print("Similarity distance:", round(result["distance"], 4))


--- Result 1 ---
Issue: hey! My Spotify randomly stops playing music and it’s happening everytime I listen... :-(
Response: Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK
Similarity distance: 0.3194

--- Result 2 ---
Issue: My Spotify keeps pausing and stops and then I go to it and can't play my music without exiting the app and coming back. Help.
Response: Hey! Can you let us know which device and operating system you're using? We'll see what we can suggest /NQ
Similarity distance: 0.3451

--- Result 3 ---
Issue: aye whenever I leave the Spotify app my music stops playing. And sometimes i will have headphones in and it’ll just stop playing all together. Please help me ;(
Response: Hey! That doesn't sound good. Can you let us know which device/OS and Spotify version you're using? We'll see what we can suggest /YM
Similarity distance: 0.3551


In [ ]:
# Add confidence level based on retrieval distance

def get_confidence(distance):
    if distance < 0.4:
        return "High"
    elif distance < 0.7:
        return "Medium"
    else:
        return "Low"

In [ ]:
# Test confidence levels

for i, result in enumerate(results, 1):
    confidence = get_confidence(result["distance"])
    
    print(f"Result {i}")
    print("Confidence:", confidence)
    print("Distance:", round(result["distance"], 4))
    print()

Result 1
Confidence: High
Distance: 0.3194

Result 2
Confidence: High
Distance: 0.3451

Result 3
Confidence: High
Distance: 0.3551



In [ ]:
results = search_similar_issues(customer_query)

print(type(results))
print(results[0])

<class 'list'>
{'customer_issue': 'hey! My Spotify randomly stops playing music and it’s happening everytime I listen... :-(', 'support_response': "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK"}


In [ ]:
def get_support_answer(customer_query):
    results = search_similar_issues(customer_query)

    if results:
        return results[0]["support_response"]
    else:
        return (
            "Sorry, I couldn't find a closely related issue. "
            "Could you please provide more details about the problem?"
        )


final_answer = get_support_answer(customer_query)

print("CUSTOMER QUERY:")
print(customer_query)

print("\nFINAL SUPPORT ANSWER:")
print(final_answer)

CUSTOMER QUERY:
Spotify stops playing music after a few songs

FINAL SUPPORT ANSWER:
Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK


In [ ]:
customer_query = "Spotify stops playing after a few songs"

answer = get_support_answer(customer_query)

print("CUSTOMER QUERY:")
print(customer_query)

print("\nSUPPORT RESPONSE:")
print(answer)

CUSTOMER QUERY:
Spotify stops playing after a few songs

SUPPORT RESPONSE:
Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK


In [ ]:
# Test with different customer queries

test_queries = [
    "Spotify is not playing any songs",
    "I cannot download songs for offline listening",
    "My Spotify Premium subscription is not working"
]

for query in test_queries:
    answer = get_support_answer(query)

    print("\n" + "=" * 60)
    print("CUSTOMER ISSUE:", query)
    print("SUPPORT RESPONSE:", answer)


CUSTOMER ISSUE: Spotify is not playing any songs
SUPPORT RESPONSE: Hey! We were off beat for a second, but we’re back on track now (excuse the puns) /CE

CUSTOMER ISSUE: I cannot download songs for offline listening
SUPPORT RESPONSE: Hi there, that doesn't sound good. Can you DM us your account's email address? We'll take a look backstage /AK

CUSTOMER ISSUE: My Spotify Premium subscription is not working
SUPPORT RESPONSE: Hey there! Could you send us a DM with your account's email address? We'll take a look backstage /MA


In [ ]:
# Save the working RAG system components

import pickle

with open("spotify_rag_model.pkl", "wb") as f:
    pickle.dump({
        "index": index,
        "ai_dataset": ai_dataset
    }, f)

print("RAG system saved successfully!")
print("Total knowledge base entries:", len(ai_dataset))

RAG system saved successfully!
Total knowledge base entries: 23815


In [54]:
def search_similar_issues(query, k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "customer_issue": ai_dataset.iloc[idx]["customer_issue"],
            "support_response": ai_dataset.iloc[idx]["support_response"],
            "distance": float(distance)
        })

    return results

In [55]:
def get_support_answer(customer_query, threshold=0.5):
    results = search_similar_issues(customer_query, k=3)

    if not results:
        return {
            "answer": "Sorry, I couldn't find a closely related issue.",
            "confidence": "Low",
            "distance": None
        }

    best_result = results[0]

    if best_result["distance"] <= threshold:
        confidence = "High"
    else:
        confidence = "Low"

    return {
        "answer": best_result["support_response"],
        "confidence": confidence,
        "distance": best_result["distance"]
    }

In [56]:
result = get_support_answer(
    "Spotify stops playing after a few songs"
)

print(result)

{'answer': "@472000 Hey Owen, that doesn't sound right! Can you let us know the device, Android, and Spotify version you're using? We'll see what we can suggest /JE", 'confidence': 'High', 'distance': 0.3106929659843445}


In [ ]:
customer_query = "Spotify stops playing after a few songs"

result = get_support_answer(customer_query)

print("CUSTOMER ISSUE:")
print(customer_query)

print("\nSUPPORT ANSWER:")
print(result["answer"])

print("\nCONFIDENCE:")
print(result["confidence"])

print("\nDISTANCE:")
print(result["distance"])

CUSTOMER ISSUE:
Spotify stops playing after a few songs

SUPPORT ANSWER:
Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK

CONFIDENCE:
High

DISTANCE:
0.3284124732017517


In [ ]:
results = search_similar_issues(customer_query, k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- RESULT {i} ---")
    print("CUSTOMER ISSUE:", result["customer_issue"])
    print("SUPPORT RESPONSE:", result["support_response"])
    print("DISTANCE:", result["distance"])


--- RESULT 1 ---
CUSTOMER ISSUE: hey! My Spotify randomly stops playing music and it’s happening everytime I listen... :-(
SUPPORT RESPONSE: Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK
DISTANCE: 0.3284124732017517

--- RESULT 2 ---
CUSTOMER ISSUE: My Spotify keeps pausing and stops and then I go to it and can't play my music without exiting the app and coming back. Help.
SUPPORT RESPONSE: Hey! Can you let us know which device and operating system you're using? We'll see what we can suggest /NQ
DISTANCE: 0.35332784056663513

--- RESULT 3 ---
CUSTOMER ISSUE: aye whenever I leave the Spotify app my music stops playing. And sometimes i will have headphones in and it’ll just stop playing all together. Please help me ;(
SUPPORT RESPONSE: Hey! That doesn't sound good. Can you let us know which device/OS and Spotify version you're using? We'll see what we can suggest /YM
DISTANCE: 0.3702150583267212


In [ ]:
test_queries = [
    "Spotify stops playing after a few songs",
    "I cannot download music for offline listening",
    "My Spotify Premium subscription is not working",
    "Spotify songs are not available"
]

for query in test_queries:
    result = get_support_answer(query)

    print("\n" + "=" * 70)
    print("CUSTOMER ISSUE:", query)
    print("SUPPORT ANSWER:", result["answer"])
    print("CONFIDENCE:", result["confidence"])
    print("DISTANCE:", result["distance"])


CUSTOMER ISSUE: Spotify stops playing after a few songs
SUPPORT ANSWER: Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK
CONFIDENCE: High
DISTANCE: 0.3284124732017517

CUSTOMER ISSUE: I cannot download music for offline listening
SUPPORT ANSWER: Hi there, that doesn't sound good. Can you DM us your account's email address? We'll take a look backstage /AK
CONFIDENCE: High
DISTANCE: 0.275166779756546

CUSTOMER ISSUE: My Spotify Premium subscription is not working
SUPPORT ANSWER: Hey there! Could you send us a DM with your account's email address? We'll take a look backstage /MA
CONFIDENCE: High
DISTANCE: 0.12932460010051727

CUSTOMER ISSUE: Spotify songs are not available
SUPPORT ANSWER: Hey there! That's not cool. Does logging out > restarting the device > logging back in help? Keep us posted /PX
CONFIDENCE: High
DISTANCE: 0.2538135051727295


In [ ]:
evaluation_results = []

for query in test_queries:
    result = get_support_answer(query)

    evaluation_results.append({
        "customer_query": query,
        "support_answer": result["answer"],
        "confidence": result["confidence"],
        "distance": result["distance"]
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,customer_query,support_answer,confidence,distance
0,Spotify stops playing after a few songs,"Hey, help's here! Can you let us know the devi...",High,0.328412
1,I cannot download music for offline listening,"Hi there, that doesn't sound good. Can you DM ...",High,0.275167
2,My Spotify Premium subscription is not working,Hey there! Could you send us a DM with your ac...,High,0.129325
3,Spotify songs are not available,Hey there! That's not cool. Does logging out >...,High,0.253814


In [ ]:
evaluation_df.to_csv(
    "spotify_rag_evaluation.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [ ]:
# View final evaluation results

print("Evaluation dataset shape:", evaluation_df.shape)

print("\nColumns:")
print(evaluation_df.columns.tolist())

print("\nFirst 10 evaluation results:")
print(evaluation_df.head(10).to_string(index=False))

print("\nMissing values:")
print(evaluation_df.isnull().sum())

Evaluation dataset shape: (4, 4)

Columns:
['customer_query', 'support_answer', 'confidence', 'distance']

First 10 evaluation results:
                                customer_query                                                                                                                          support_answer confidence  distance
       Spotify stops playing after a few songs Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK       High  0.328412
 I cannot download music for offline listening                          Hi there, that doesn't sound good. Can you DM us your account's email address? We'll take a look backstage /AK       High  0.275167
My Spotify Premium subscription is not working                                    Hey there! Could you send us a DM with your account's email address? We'll take a look backstage /MA       High  0.129325
               Spotify songs are not available  

In [ ]:
# ================================
# 1. LOAD AND CHECK EVALUATION FILE
# ================================

import pandas as pd

evaluation_df = pd.read_csv("spotify_rag_evaluation.csv")

print("Evaluation dataset shape:", evaluation_df.shape)
print("\nColumns:")
print(evaluation_df.columns.tolist())

print("\nFirst 10 rows:")
print(evaluation_df.head(10).to_string(index=False))

print("\nMissing values:")
print(evaluation_df.isnull().sum())

Evaluation dataset shape: (4, 4)

Columns:
['customer_query', 'support_answer', 'confidence', 'distance']

First 10 rows:
                                customer_query                                                                                                                          support_answer confidence  distance
       Spotify stops playing after a few songs Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK       High  0.328412
 I cannot download music for offline listening                          Hi there, that doesn't sound good. Can you DM us your account's email address? We'll take a look backstage /AK       High  0.275167
My Spotify Premium subscription is not working                                    Hey there! Could you send us a DM with your account's email address? We'll take a look backstage /MA       High  0.129325
               Spotify songs are not available                

In [ ]:
# ================================
# 2. AUTOMATIC NUMERICAL SUMMARY
# ================================

numeric_columns = evaluation_df.select_dtypes(include="number").columns.tolist()

if numeric_columns:
    print("Numerical Evaluation Summary:")
    print(evaluation_df[numeric_columns].describe())
else:
    print("No numerical evaluation columns found.")

Numerical Evaluation Summary:
       distance
count  4.000000
mean   0.246679
std    0.084291
min    0.129325
25%    0.222691
50%    0.264490
75%    0.288478
max    0.328412


In [ ]:
# ================================
# 3. CHECK DUPLICATES
# ================================

duplicate_rows = evaluation_df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

if duplicate_rows == 0:
    print("Evaluation dataset has no duplicate rows.")
else:
    evaluation_df = evaluation_df.drop_duplicates()
    print("Duplicates removed.")

Duplicate rows: 0
Evaluation dataset has no duplicate rows.


In [ ]:
# ================================
# 4. FINAL EVALUATION SUMMARY
# ================================

print("=" * 50)
print("SPOTIFY RAG EVALUATION SUMMARY")
print("=" * 50)

print("Total evaluation records:", len(evaluation_df))
print("Total columns:", len(evaluation_df.columns))
print("Missing values:", evaluation_df.isnull().sum().sum())
print("Duplicate rows:", evaluation_df.duplicated().sum())

print("\nEvaluation completed successfully!")

SPOTIFY RAG EVALUATION SUMMARY
Total evaluation records: 4
Total columns: 4
Missing values: 0
Duplicate rows: 0

Evaluation completed successfully!


In [ ]:
# ================================
# 5. SAVE FINAL CLEAN EVALUATION
# ================================

evaluation_df.to_csv(
    "spotify_rag_final_evaluation.csv",
    index=False
)

print("Final evaluation file saved successfully!")
print("Final shape:", evaluation_df.shape)

Final evaluation file saved successfully!
Final shape: (4, 4)


In [ ]:
# FINAL PROJECT TEST CASES

test_queries = [
    "Spotify stops playing music",
    "I cannot listen to songs on Spotify",
    "My Spotify premium is not working",
    "How do I delete a playlist?",
    "Spotify songs are not available"
]

for i, query in enumerate(test_queries, 1):
    answer = get_support_answer(query)

    print(f"\n--- TEST CASE {i} ---")
    print("Customer Issue:", query)
    print("Support Response:", answer)


--- TEST CASE 1 ---
Customer Issue: Spotify stops playing music
Support Response: {'answer': "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK", 'confidence': 'High', 'distance': 0.28929996490478516}

--- TEST CASE 2 ---
Customer Issue: I cannot listen to songs on Spotify
Support Response: {'answer': "Hey! Help's here. Are you getting any specific errors? If so, a screenshot will be handy /CP", 'confidence': 'High', 'distance': 0.2269885092973709}

--- TEST CASE 3 ---
Customer Issue: My Spotify premium is not working
Support Response: {'answer': "Hi, we've just replied to your DM. We'll carry on chatting there /SY", 'confidence': 'High', 'distance': 0.032288990914821625}

--- TEST CASE 4 ---
Customer Issue: How do I delete a playlist?
Support Response: {'answer': "Hi there, the cavalry's here! If you'd like to remove a track from your playlist, we'd recommend following the steps at Hope this helps /

In [ ]:
# ================================
# 8. SAVE TEST RESULTS
# ================================

test_results = []

for query in test_queries:
    answer = get_support_answer(query)

    test_results.append({
        "customer_issue": query,
        "generated_support_response": answer
    })

test_results_df = pd.DataFrame(test_results)

test_results_df.to_csv(
    "spotify_rag_test_results.csv",
    index=False
)

print("Test results saved successfully!")
print(test_results_df)

Test results saved successfully!
                        customer_issue  \
0          Spotify stops playing music   
1  I cannot listen to songs on Spotify   
2    My Spotify premium is not working   
3          How do I delete a playlist?   
4      Spotify songs are not available   

                          generated_support_response  
0  {'answer': 'Hey, help's here! Can you let us k...  
1  {'answer': 'Hey! Help's here. Are you getting ...  
2  {'answer': 'Hi, we've just replied to your DM....  
3  {'answer': 'Hi there, the cavalry's here! If y...  
4  {'answer': 'Hey there! That's not cool. Does l...  


In [ ]:
# FINAL PROJECT TEST CASES AND SAVE RESULTS

test_queries = [
    "Spotify stops playing music",
    "I cannot listen to songs on Spotify",
    "My Spotify premium is not working",
    "How do I delete a playlist?",
    "Spotify songs are not available"
]

test_results = []

for i, query in enumerate(test_queries, 1):
    answer = get_support_answer(query)

    test_results.append({
        "test_id": i,
        "customer_issue": query,
        "generated_support_response": answer
    })

    print(f"\n--- TEST CASE {i} ---")
    print("Customer Issue:", query)
    print("Support Response:", answer)

test_results_df = pd.DataFrame(test_results)

test_results_df.to_csv(
    "spotify_rag_test_results.csv",
    index=False
)

print("\nTest results saved successfully!")
print("Final test results shape:", test_results_df.shape)


--- TEST CASE 1 ---
Customer Issue: Spotify stops playing music
Support Response: {'answer': "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK", 'confidence': 'High', 'distance': 0.28929996490478516}

--- TEST CASE 2 ---
Customer Issue: I cannot listen to songs on Spotify
Support Response: {'answer': "Hey! Help's here. Are you getting any specific errors? If so, a screenshot will be handy /CP", 'confidence': 'High', 'distance': 0.2269885092973709}

--- TEST CASE 3 ---
Customer Issue: My Spotify premium is not working
Support Response: {'answer': "Hi, we've just replied to your DM. We'll carry on chatting there /SY", 'confidence': 'High', 'distance': 0.032288990914821625}

--- TEST CASE 4 ---
Customer Issue: How do I delete a playlist?
Support Response: {'answer': "Hi there, the cavalry's here! If you'd like to remove a track from your playlist, we'd recommend following the steps at Hope this helps /

In [ ]:
print(test_results_df.to_string(index=False))

print("\nMissing values:")
print(test_results_df.isnull().sum())

print("\nTotal test cases:", len(test_results_df))

 test_id                      customer_issue                                                                                                                                                                                       generated_support_response
       1         Spotify stops playing music     {'answer': 'Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK', 'confidence': 'High', 'distance': 0.28929996490478516}
       2 I cannot listen to songs on Spotify                                                 {'answer': 'Hey! Help's here. Are you getting any specific errors? If so, a screenshot will be handy /CP', 'confidence': 'High', 'distance': 0.2269885092973709}
       3   My Spotify premium is not working                                                                       {'answer': 'Hi, we've just replied to your DM. We'll carry on chatting there /SY', 'confidence': 'High', 'distance': 0.0322

In [ ]:
print("=" * 60)
print("SPOTIFY CUSTOMER SUPPORT RAG - FINAL STATISTICS")
print("=" * 60)

print("Knowledge base pairs:", len(ai_dataset))
print("Evaluation records:", len(evaluation_df))
print("Test cases:", len(test_results_df))
print("Embedding model:", model.__class__.__name__)
print("Retrieval system: FAISS vector similarity search")
print("Final status: COMPLETED")

SPOTIFY CUSTOMER SUPPORT RAG - FINAL STATISTICS
Knowledge base pairs: 23815
Evaluation records: 4
Test cases: 5
Embedding model: SentenceTransformer
Retrieval system: FAISS vector similarity search
Final status: COMPLETED


In [ ]:
project_summary = pd.DataFrame({
    "metric": [
        "Knowledge Base Size",
        "Evaluation Records",
        "Test Cases",
        "Embedding Model",
        "Retrieval Method"
    ],
    "value": [
        len(ai_dataset),
        len(evaluation_df),
        len(test_results_df),
        model.__class__.__name__,
        "FAISS Vector Similarity Search"
    ]
})

project_summary.to_csv(
    "spotify_rag_project_summary.csv",
    index=False
)

print("Project summary saved successfully!")
print(project_summary)

Project summary saved successfully!
                metric                           value
0  Knowledge Base Size                           23815
1   Evaluation Records                               4
2           Test Cases                               5
3      Embedding Model             SentenceTransformer
4     Retrieval Method  FAISS Vector Similarity Search


In [ ]:
import os

output_files = [
    "spotify_ai_final_dataset.csv",
    "spotify_rag_evaluation.csv",
    "spotify_rag_final_evaluation.csv",
    "spotify_rag_test_results.csv",
    "spotify_rag_project_summary.csv"
]

for file in output_files:
    print(f"{file}: {'FOUND ✓' if os.path.exists(file) else 'NOT FOUND ✗'}")

spotify_ai_final_dataset.csv: FOUND ✓
spotify_rag_evaluation.csv: FOUND ✓
spotify_rag_final_evaluation.csv: FOUND ✓
spotify_rag_test_results.csv: FOUND ✓
spotify_rag_project_summary.csv: FOUND ✓


In [ ]:
saved_test_results = pd.read_csv("spotify_rag_test_results.csv")

print("Saved test results shape:", saved_test_results.shape)
print()

print(saved_test_results.to_string(index=False))

Saved test results shape: (5, 3)

 test_id                      customer_issue                                                                                                                                                                                       generated_support_response
       1         Spotify stops playing music     {'answer': "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK", 'confidence': 'High', 'distance': 0.28929996490478516}
       2 I cannot listen to songs on Spotify                                                 {'answer': "Hey! Help's here. Are you getting any specific errors? If so, a screenshot will be handy /CP", 'confidence': 'High', 'distance': 0.2269885092973709}
       3   My Spotify premium is not working                                                                       {'answer': "Hi, we've just replied to your DM. We'll carry on chatting there /SY", 'confi

In [ ]:
final_metrics = {
    "Knowledge base pairs": len(ai_dataset),
    "Evaluation records": len(evaluation_df),
    "Test cases": len(test_results_df),
    "Test results saved": os.path.exists("spotify_rag_test_results.csv"),
    "FAISS index available": "index" in globals(),
    "Embedding model available": "model" in globals()
}

for metric, value in final_metrics.items():
    print(f"{metric}: {value}")

Knowledge base pairs: 23815
Evaluation records: 4
Test cases: 5
Test results saved: True
FAISS index available: True
Embedding model available: True


In [ ]:
final_metrics_df = pd.DataFrame(
    list(final_metrics.items()),
    columns=["Metric", "Value"]
)

final_metrics_df.to_csv(
    "spotify_rag_final_metrics.csv",
    index=False
)

print("Final metrics saved successfully!")
print(final_metrics_df)

Final metrics saved successfully!
                      Metric  Value
0       Knowledge base pairs  23815
1         Evaluation records      4
2                 Test cases      5
3         Test results saved   True
4      FAISS index available   True
5  Embedding model available   True


In [ ]:
print("=" * 60)
print("SPOTIFY CUSTOMER SUPPORT RAG - COMPLETION CHECK")
print("=" * 60)

print("Dataset preparation: COMPLETED")
print("Text cleaning: COMPLETED")
print("Customer-support pairing: COMPLETED")
print("Embeddings: COMPLETED")
print("FAISS retrieval: COMPLETED")
print("Support answer generation: COMPLETED")
print("Evaluation: COMPLETED")
print("Testing: COMPLETED")
print("Results saved: COMPLETED")

print("\nPROJECT IMPLEMENTATION COMPLETED SUCCESSFULLY!")

SPOTIFY CUSTOMER SUPPORT RAG - COMPLETION CHECK
Dataset preparation: COMPLETED
Text cleaning: COMPLETED
Customer-support pairing: COMPLETED
Embeddings: COMPLETED
FAISS retrieval: COMPLETED
Support answer generation: COMPLETED
Evaluation: COMPLETED
Testing: COMPLETED
Results saved: COMPLETED

PROJECT IMPLEMENTATION COMPLETED SUCCESSFULLY!


In [ ]:
import uuid
from datetime import datetime

def create_ticket(customer_query):
    
    results = search_similar_issues(customer_query, k=3)
    
    best_result = results[0]
    
    distance = best_result["distance"]
    
    if distance < 0.4:
        confidence = "High"
        status = "Resolved Automatically"
    elif distance < 0.7:
        confidence = "Medium"
        status = "Needs Review"
    else:
        confidence = "Low"
        status = "Escalated"
    
    ticket = {
        "ticket_id": str(uuid.uuid4())[:8],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "customer_issue": customer_query,
        "suggested_response": best_result["support_response"],
        "confidence": confidence,
        "distance": round(distance, 4),
        "status": status
    }
    
    return ticket

In [ ]:
ticket = create_ticket(
    "Spotify stops playing after a few songs"
)

for key, value in ticket.items():
    print(f"{key}: {value}")

ticket_id: eb9128db
created_at: 2026-09-11 21:08:05
customer_issue: Spotify stops playing after a few songs
suggested_response: Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK
confidence: High
distance: 0.3284
status: Resolved Automatically


In [ ]:
ticket_queries = [
    "Spotify stops playing music",
    "I cannot download songs",
    "My Spotify Premium is not working",
    "I cannot access my account",
    "Spotify keeps crashing"
]

tickets = []

for query in ticket_queries:
    tickets.append(create_ticket(query))

tickets_df = pd.DataFrame(tickets)

tickets_df

,ticket_id,created_at,customer_issue,suggested_response,confidence,distance,status
0,f363d97d,2026-09-11 21:08:20,Spotify stops playing music,"Hey, help's here! Can you let us know the devi...",High,0.2893,Resolved Automatically
1,b0e41f58,2026-09-11 21:08:20,I cannot download songs,"Hey, help's here! Can you DM us your account's...",High,0.2390,Resolved Automatically
2,7f71fbb2,2026-09-11 21:08:20,My Spotify Premium is not working,"Hi, we've just replied to your DM. We'll carry...",High,0.0323,Resolved Automatically
3,a24240ca,2026-09-11 21:08:20,I cannot access my account,Hey there! Could you DM us your account's user...,High,0.2887,Resolved Automatically
4,d56d6206,2026-09-11 21:08:21,Spotify keeps crashing,Hey Stephen! That's not cool. What operating s...,High,0.2370,Resolved Automatically


In [ ]:
tickets_df.to_csv(
    "spotify_generated_tickets.csv",
    index=False
)

print("Generated tickets saved successfully!")
print("Total tickets:", len(tickets_df))

Generated tickets saved successfully!
Total tickets: 5


In [ ]:
print(
    tickets_df["status"]
    .value_counts()
)

status
Resolved Automatically    5
Name: count, dtype: int64


In [ ]:
# 1. Add escalation decision to each ticket

def add_ticket_action(ticket):
    
    if ticket["confidence"] == "High":
        ticket["action"] = "Auto Respond"
        
    elif ticket["confidence"] == "Medium":
        ticket["action"] = "Human Review"
        
    else:
        ticket["action"] = "Escalate to Support Agent"
    
    return ticket

In [ ]:
# 2. Apply actions to all generated tickets

updated_tickets = []

for query in ticket_queries:
    ticket = create_ticket(query)
    ticket = add_ticket_action(ticket)
    updated_tickets.append(ticket)

updated_tickets_df = pd.DataFrame(updated_tickets)

updated_tickets_df

,ticket_id,created_at,customer_issue,suggested_response,confidence,distance,status,action
0,a484c861,2026-09-11 21:09:49,Spotify stops playing music,"Hey, help's here! Can you let us know the devi...",High,0.2893,Resolved Automatically,Auto Respond
1,faeb605f,2026-09-11 21:09:49,I cannot download songs,"Hey, help's here! Can you DM us your account's...",High,0.2390,Resolved Automatically,Auto Respond
2,8d9b3a9b,2026-09-11 21:09:49,My Spotify Premium is not working,"Hi, we've just replied to your DM. We'll carry...",High,0.0323,Resolved Automatically,Auto Respond
3,f034c7a5,2026-09-11 21:09:49,I cannot access my account,Hey there! Could you DM us your account's user...,High,0.2887,Resolved Automatically,Auto Respond
4,239fefc5,2026-09-11 21:09:49,Spotify keeps crashing,Hey Stephen! That's not cool. What operating s...,High,0.2370,Resolved Automatically,Auto Respond


In [ ]:
# 3. Check agent actions

print(
    updated_tickets_df[
        ["customer_issue", "confidence", "status", "action"]
    ].to_string(index=False)
)

                   customer_issue confidence                 status       action
      Spotify stops playing music       High Resolved Automatically Auto Respond
          I cannot download songs       High Resolved Automatically Auto Respond
My Spotify Premium is not working       High Resolved Automatically Auto Respond
       I cannot access my account       High Resolved Automatically Auto Respond
           Spotify keeps crashing       High Resolved Automatically Auto Respond


In [ ]:
# 4. Save final agent decisions

updated_tickets_df.to_csv(
    "spotify_agent_ticket_results.csv",
    index=False
)

print("Agent ticket results saved successfully!")

Agent ticket results saved successfully!


In [ ]:
# 5. Final agent workflow summary

print("=" * 70)
print("AGENTIC AI TICKET SYSTEM WORKFLOW")
print("=" * 70)

print("""
1. Customer submits issue
        ↓
2. AI converts issue into embedding
        ↓
3. FAISS retrieves similar historical issues
        ↓
4. System calculates retrieval distance
        ↓
5. AI determines confidence
        ↓
6. High Confidence → Auto Respond
   Medium Confidence → Human Review
   Low Confidence → Escalate
        ↓
7. Structured ticket is created and saved
""")

AGENTIC AI TICKET SYSTEM WORKFLOW

1. Customer submits issue
        ↓
2. AI converts issue into embedding
        ↓
3. FAISS retrieves similar historical issues
        ↓
4. System calculates retrieval distance
        ↓
5. AI determines confidence
        ↓
6. High Confidence → Auto Respond
   Medium Confidence → Human Review
   Low Confidence → Escalate
        ↓
7. Structured ticket is created and saved



In [ ]:
# 1. Assign priority based on confidence and keywords

def assign_priority(ticket):
    
    issue = ticket["customer_issue"].lower()
    
    urgent_keywords = [
        "account", "payment", "premium",
        "charged", "cannot access", "hacked"
    ]
    
    if any(keyword in issue for keyword in urgent_keywords):
        ticket["priority"] = "High"
        
    elif ticket["confidence"] == "Low":
        ticket["priority"] = "High"
        
    elif ticket["confidence"] == "Medium":
        ticket["priority"] = "Medium"
        
    else:
        ticket["priority"] = "Low"
    
    return ticket

In [ ]:
# 2. Apply priority to all tickets

final_tickets = []

for query in ticket_queries:
    ticket = create_ticket(query)
    ticket = add_ticket_action(ticket)
    ticket = assign_priority(ticket)
    
    final_tickets.append(ticket)

final_tickets_df = pd.DataFrame(final_tickets)

final_tickets_df

,ticket_id,created_at,customer_issue,suggested_response,confidence,distance,status,action,priority
0,b4cfce8d,2026-09-11 21:11:11,Spotify stops playing music,"Hey, help's here! Can you let us know the devi...",High,0.2893,Resolved Automatically,Auto Respond,Low
1,18ba728c,2026-09-11 21:11:11,I cannot download songs,"Hey, help's here! Can you DM us your account's...",High,0.2390,Resolved Automatically,Auto Respond,Low
2,563df867,2026-09-11 21:11:11,My Spotify Premium is not working,"Hi, we've just replied to your DM. We'll carry...",High,0.0323,Resolved Automatically,Auto Respond,High
3,c08d63ab,2026-09-11 21:11:11,I cannot access my account,Hey there! Could you DM us your account's user...,High,0.2887,Resolved Automatically,Auto Respond,High
4,80d5d341,2026-09-11 21:11:11,Spotify keeps crashing,Hey Stephen! That's not cool. What operating s...,High,0.2370,Resolved Automatically,Auto Respond,Low


In [ ]:
# 3. View final ticket system output

print(
    final_tickets_df[
        [
            "ticket_id",
            "customer_issue",
            "confidence",
            "priority",
            "status",
            "action"
        ]
    ].to_string(index=False)
)

ticket_id                    customer_issue confidence priority                 status       action
 b4cfce8d       Spotify stops playing music       High      Low Resolved Automatically Auto Respond
 18ba728c           I cannot download songs       High      Low Resolved Automatically Auto Respond
 563df867 My Spotify Premium is not working       High     High Resolved Automatically Auto Respond
 c08d63ab        I cannot access my account       High     High Resolved Automatically Auto Respond
 80d5d341            Spotify keeps crashing       High      Low Resolved Automatically Auto Respond


In [ ]:
# 4. Save final tickets

final_tickets_df.to_csv(
    "spotify_final_tickets.csv",
    index=False
)

print("Final tickets saved successfully!")
print("Total tickets:", len(final_tickets_df))

Final tickets saved successfully!
Total tickets: 5


In [ ]:
# 5. Ticket priority summary

print("Priority Summary:")
print(final_tickets_df["priority"].value_counts())

print("\nAction Summary:")
print(final_tickets_df["action"].value_counts())

print("\nStatus Summary:")
print(final_tickets_df["status"].value_counts())

Priority Summary:
priority
Low     3
High    2
Name: count, dtype: int64

Action Summary:
action
Auto Respond    5
Name: count, dtype: int64

Status Summary:
status
Resolved Automatically    5
Name: count, dtype: int64


In [ ]:
# 1. Define ticket categories

def classify_category(customer_issue):
    
    issue = customer_issue.lower()
    
    categories = {
        "Playback": ["not playing", "stops playing", "music stops", "songs"],
        "Premium": ["premium", "subscription", "upgrade"],
        "Account": ["account", "login", "sign in", "access"],
        "Download": ["download", "offline"],
        "Payment": ["payment", "charged", "billing", "refund"],
        "Playlist": ["playlist"],
        "App Issue": ["crash", "app", "error", "update"]
    }
    
    for category, keywords in categories.items():
        if any(keyword in issue for keyword in keywords):
            return category
    
    return "Other"

In [ ]:
# 2. Apply category classification to all tickets

final_tickets_df["category"] = final_tickets_df[
    "customer_issue"
].apply(classify_category)

print(
    final_tickets_df[
        [
            "ticket_id",
            "customer_issue",
            "category",
            "priority",
            "action"
        ]
    ].to_string(index=False)
)

ticket_id                    customer_issue  category priority       action
 b4cfce8d       Spotify stops playing music  Playback      Low Auto Respond
 18ba728c           I cannot download songs  Playback      Low Auto Respond
 563df867 My Spotify Premium is not working   Premium     High Auto Respond
 c08d63ab        I cannot access my account   Account     High Auto Respond
 80d5d341            Spotify keeps crashing App Issue      Low Auto Respond


In [ ]:
# 3. Save updated tickets

final_tickets_df.to_csv(
    "spotify_final_tickets.csv",
    index=False
)

print("Updated final tickets saved successfully!")

Updated final tickets saved successfully!


In [ ]:
# 4. Check category distribution

print("Ticket Category Summary:\n")

print(
    final_tickets_df["category"]
    .value_counts()
)

Ticket Category Summary:

category
Playback     2
Premium      1
Account      1
App Issue    1
Name: count, dtype: int64


In [ ]:
# 5. Final complete ticket output

print("=" * 80)

for _, ticket in final_tickets_df.iterrows():
    
    print("\nTICKET ID:", ticket["ticket_id"])
    print("ISSUE:", ticket["customer_issue"])
    print("CATEGORY:", ticket["category"])
    print("PRIORITY:", ticket["priority"])
    print("CONFIDENCE:", ticket["confidence"])
    print("ACTION:", ticket["action"])
    print("STATUS:", ticket["status"])
    
    print("-" * 80)


TICKET ID: b4cfce8d
ISSUE: Spotify stops playing music
CATEGORY: Playback
PRIORITY: Low
CONFIDENCE: High
ACTION: Auto Respond
STATUS: Resolved Automatically
--------------------------------------------------------------------------------

TICKET ID: 18ba728c
ISSUE: I cannot download songs
CATEGORY: Playback
PRIORITY: Low
CONFIDENCE: High
ACTION: Auto Respond
STATUS: Resolved Automatically
--------------------------------------------------------------------------------

TICKET ID: 563df867
ISSUE: My Spotify Premium is not working
CATEGORY: Premium
PRIORITY: High
CONFIDENCE: High
ACTION: Auto Respond
STATUS: Resolved Automatically
--------------------------------------------------------------------------------

TICKET ID: c08d63ab
ISSUE: I cannot access my account
CATEGORY: Account
PRIORITY: High
CONFIDENCE: High
ACTION: Auto Respond
STATUS: Resolved Automatically
--------------------------------------------------------------------------------

TICKET ID: 80d5d341
ISSUE: Spotify keeps c

In [ ]:
# 1. Create ticket analytics summary

analytics = {
    "Total Tickets": len(final_tickets_df),
    "High Priority": (final_tickets_df["priority"] == "High").sum(),
    "Medium Priority": (final_tickets_df["priority"] == "Medium").sum(),
    "Low Priority": (final_tickets_df["priority"] == "Low").sum(),
    "Auto Respond": (final_tickets_df["action"] == "Auto Respond").sum(),
    "Human Review": (final_tickets_df["action"] == "Human Review").sum(),
    "Escalated": (final_tickets_df["action"] == "Escalate to Support Agent").sum()
}

for key, value in analytics.items():
    print(f"{key}: {value}")

Total Tickets: 5
High Priority: 2
Medium Priority: 0
Low Priority: 3
Auto Respond: 5
Human Review: 0
Escalated: 0


In [ ]:
# 2. Category analytics

category_summary = (
    final_tickets_df["category"]
    .value_counts()
    .reset_index()
)

category_summary.columns = ["category", "ticket_count"]

print(category_summary)

    category  ticket_count
0   Playback             2
1    Premium             1
2    Account             1
3  App Issue             1


In [ ]:
# 3. Priority analytics

priority_summary = (
    final_tickets_df["priority"]
    .value_counts()
    .reset_index()
)

priority_summary.columns = ["priority", "ticket_count"]

print(priority_summary)

  priority  ticket_count
0      Low             3
1     High             2


In [ ]:
# 4. Action analytics

action_summary = (
    final_tickets_df["action"]
    .value_counts()
    .reset_index()
)

action_summary.columns = ["action", "ticket_count"]

print(action_summary)

         action  ticket_count
0  Auto Respond             5


In [ ]:
# 5. Save all analytics

analytics_df = pd.DataFrame(
    list(analytics.items()),
    columns=["metric", "value"]
)

analytics_df.to_csv("ticket_analytics.csv", index=False)
category_summary.to_csv("ticket_category_summary.csv", index=False)
priority_summary.to_csv("ticket_priority_summary.csv", index=False)
action_summary.to_csv("ticket_action_summary.csv", index=False)

print("All ticket analytics saved successfully!")

All ticket analytics saved successfully!


In [ ]:
!pip install fastapi uvicorn nest-asyncio


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import faiss

faiss.write_index(index, "spotify_faiss_index.faiss")

print("FAISS index saved successfully!")

NameError: name 'index' is not defined

In [ ]:
ai_dataset.to_pickle("spotify_ai_dataset.pkl")

print("AI dataset saved successfully!")

NameError: name 'ai_dataset' is not defined

In [ ]:
%%writefile app.py

import pandas as pd
import faiss
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from datetime import datetime
import uuid

app = FastAPI(title="Agentic AI Ticket System")

model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index("spotify_faiss_index.faiss")

ai_dataset = pd.read_pickle("spotify_ai_dataset.pkl")


class TicketRequest(BaseModel):
    customer_issue: str


def search_similar_issues(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "customer_issue": ai_dataset.iloc[idx]["customer_issue"],
            "support_response": ai_dataset.iloc[idx]["support_response"],
            "distance": float(distance)
        })

    return results


@app.get("/")
def home():
    return {
        "message": "Agentic AI Ticket System API is running"
    }


@app.post("/create-ticket")
def create_ticket(request: TicketRequest):

    results = search_similar_issues(
        request.customer_issue,
        k=3
    )

    best_result = results[0]
    distance = best_result["distance"]

    if distance < 0.4:
        confidence = "High"
        action = "Auto Respond"
        status = "Resolved Automatically"

    elif distance < 0.7:
        confidence = "Medium"
        action = "Human Review"
        status = "Needs Review"

    else:
        confidence = "Low"
        action = "Escalate to Support Agent"
        status = "Escalated"

    return {
        "ticket_id": str(uuid.uuid4())[:8],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "customer_issue": request.customer_issue,
        "suggested_response": best_result["support_response"],
        "confidence": confidence,
        "distance": round(distance, 4),
        "action": action,
        "status": status
    }

Writing app.py


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(target=run_server)
thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


Exception in thread Thread-9 (run_server):
Traceback (most recent call last):
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run


    self._target(*self._args, **self._kwargs)
  File "C:\Users\Anitha Kengeri\AppData\Local\Temp\ipykernel_24220\2012371060.py", line 9, in run_server
NameError: name 'app' is not defined


In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/create-ticket",
    json={
        "customer_issue": "Spotify stops playing after a few songs"
    }
)

print(response.status_code)
print(response.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /create-ticket (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000013F020D1DB0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
%run app.py

print("app.py loaded successfully!")
print(app)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 700.70it/s]


app.py loaded successfully!


In [ ]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000
    )

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(2)

print("FastAPI server started successfully!")

Exception in thread Thread-10 (run_server):
Traceback (most recent call last):
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Anitha Kengeri\AppData\Local\Temp\ipykernel_24220\3012549068.py", line 9, in run_server
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\uvicorn\main.py", line 621, in run
    server.run()
  File "c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\uvicorn\server.py", line 77, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_

FastAPI server started successfully!


In [ ]:
!pip uninstall -y websockets
!pip install websockets==12.0

Found existing installation: websockets 10.4
Uninstalling websockets-10.4:
  Successfully uninstalled websockets-10.4


You can safely remove it manually.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasa 3.6.21 requires attrs<22.2,>=19.3, but you have attrs 25.3.0 which is incompatible.
rasa 3.6.21 requires jsonschema<4.18,>=3.2, but you have jsonschema 4.25.1 which is incompatible.
rasa 3.6.21 requires numpy<1.25.0,>=1.19.2; python_version >= "3.8" and python_version < "3.11", but you have numpy 1.26.4 which is incompatible.
rasa 3.6.21 requires packaging<21.0,>=20.0, but you have packaging 25.0 which is incompatible.
rasa 3.6.21 requires prompt-toolkit<3.0.29,>=3.0, but you have prompt-toolkit 3.0.52 which is incompatible.
rasa 3.6.21 requires pydantic<1.10.10, but you have pydantic 2.13.3 which is incompatible.
rasa 3.6.21 requires regex<2022.11,>=2020.6, but you have regex 2026.4.4 which is incompatible.
rasa 3.6.21 requires scikit-learn<1.2,>=0.22; python_version >= "3.8" and python_version < "3.11", but

In [ ]:
%run app.py

print("app.py loaded successfully!")
print(app)

c:\Users\Anitha Kengeri\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1512.77it/s]


app.py loaded successfully!


In [ ]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000
    )

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(2)

print("FastAPI server started successfully!")

INFO:     Started server process [17932]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


FastAPI server started successfully!


INFO:     127.0.0.1:63896 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63898 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63899 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63900 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63902 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63903 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:63904 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58350 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58351 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58352 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58353 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58354 - "POST /create-ticket HTTP/1.1" 200 OK
INFO:     127.0.0.1:58362 - "POST /create-ticket HTTP/1.1" 200 OK


In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/create-ticket",
    json={
        "customer_issue": "Spotify stops playing after a few songs"
    }
)

print("Status Code:", response.status_code)
print("\nResponse:")
print(response.json())

Status Code: 200

Response:
{'ticket_id': 'b9c45891', 'created_at': '2026-09-12 20:29:43', 'customer_issue': 'Spotify stops playing after a few songs', 'suggested_response': "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK", 'confidence': 'High', 'distance': 0.3284, 'action': 'Auto Respond', 'status': 'Resolved Automatically'}


In [ ]:
import requests
import pandas as pd

test_queries = [
    "Spotify stops playing after a few songs",
    "I cannot play songs on Spotify",
    "My Spotify account is not working"
]

test_results = []

for query in test_queries:
    response = requests.post(
        "http://127.0.0.1:8000/create-ticket",
        json={"customer_issue": query}
    )
    
    result = response.json()
    result["customer_issue"] = query
    result["status_code"] = response.status_code
    
    test_results.append(result)

test_df = pd.DataFrame(test_results)

test_df.to_csv(
    "ticket_api_test_results.csv",
    index=False
)

print("API test results saved successfully!")
print(test_df)

API test results saved successfully!
  ticket_id           created_at                           customer_issue  \
0  5dd7f4f0  2026-09-12 20:29:52  Spotify stops playing after a few songs   
1  62f22327  2026-09-12 20:29:52           I cannot play songs on Spotify   
2  d46a7db0  2026-09-12 20:29:52        My Spotify account is not working   

                                  suggested_response confidence  distance  \
0  Hey, help's here! Can you let us know the devi...       High    0.3284   
1  Hey Celia, that doesn't sound good. Can you le...       High    0.2137   
2  Hi Diyana! We've just sent a DM your way. Let'...       High    0.2290   

         action                  status  status_code  
0  Auto Respond  Resolved Automatically          200  
1  Auto Respond  Resolved Automatically          200  
2  Auto Respond  Resolved Automatically          200  


In [ ]:
# Create and save API test summary

api_summary = {
    "total_tests": len(test_df),
    "successful_requests": int((test_df["status_code"] == 200).sum()),
    "failed_requests": int((test_df["status_code"] != 200).sum()),
    "high_confidence_results": int((test_df["confidence"] == "High").sum()),
    "auto_resolved_tickets": int(
        (test_df["status"] == "Resolved Automatically").sum()
    )
}

summary_df = pd.DataFrame(
    list(api_summary.items()),
    columns=["metric", "value"]
)

summary_df.to_csv(
    "ticket_api_test_summary.csv",
    index=False
)

print("API TEST SUMMARY")
print("=" * 40)
print(summary_df.to_string(index=False))

print("\nSummary saved successfully!")

API TEST SUMMARY
                 metric  value
            total_tests      3
    successful_requests      3
        failed_requests      0
high_confidence_results      3
  auto_resolved_tickets      3

Summary saved successfully!


In [ ]:
print(test_df.columns.tolist())
print(test_df.head())

['ticket_id', 'created_at', 'customer_issue', 'suggested_response', 'confidence', 'distance', 'action', 'status', 'status_code']
  ticket_id           created_at                           customer_issue  \
0  5dd7f4f0  2026-09-12 20:29:52  Spotify stops playing after a few songs   
1  62f22327  2026-09-12 20:29:52           I cannot play songs on Spotify   
2  d46a7db0  2026-09-12 20:29:52        My Spotify account is not working   

                                  suggested_response confidence  distance  \
0  Hey, help's here! Can you let us know the devi...       High    0.3284   
1  Hey Celia, that doesn't sound good. Can you le...       High    0.2137   
2  Hi Diyana! We've just sent a DM your way. Let'...       High    0.2290   

         action                  status  status_code  
0  Auto Respond  Resolved Automatically          200  
1  Auto Respond  Resolved Automatically          200  
2  Auto Respond  Resolved Automatically          200  


In [ ]:
# STEP 5 — Validate API test results

required_columns = [
    "id",
    "created_at",
    "customer_issue",
    "suggested_response",
    "confidence",
    "distance",
    "action",
    "status",
    "status_code"
]

missing_columns = [
    col for col in required_columns
    if col not in test_df.columns
]
print("VALIDATION RESULTS")
print("=" * 40)

print("Total tests:", len(test_df))
print("Missing columns:", missing_columns)

print(
    "Successful requests:",
    (test_df["status_code"] == 200).sum()
)

print(
    "Empty responses:",
    test_df["suggested_response"].isna().sum()
)

print(
    "Duplicate ticket IDs:",
    test_df["ticket_id"].duplicated().sum()
)

VALIDATION RESULTS
Total tests: 3
Missing columns: ['id']
Successful requests: 3
Empty responses: 0
Duplicate ticket IDs: 0


In [ ]:
# STEP 5 — Validate API test results

print("VALIDATION RESULTS")
print("=" * 40)

print("Total tests:", len(test_df))
print("Available columns:", test_df.columns.tolist())

print(
    "Successful requests:",
    (test_df["status_code"] == 200).sum()
)

print(
    "Empty responses:",
    test_df["suggested_response"].isna().sum()
)

print(
    "Duplicate customer issues:",
    test_df["customer_issue"].duplicated().sum()
)

print(
    "Empty customer issues:",
    test_df["customer_issue"].isna().sum()
)

VALIDATION RESULTS
Total tests: 3
Available columns: ['ticket_id', 'created_at', 'customer_issue', 'suggested_response', 'confidence', 'distance', 'action', 'status', 'status_code']
Successful requests: 3
Empty responses: 0
Duplicate customer issues: 0
Empty customer issues: 0


In [ ]:
# STEP 6 — Test API with additional customer issues

additional_queries = [
    "Spotify keeps crashing",
    "I forgot my Spotify password",
    "My playlist disappeared",
    "Music is buffering continuously",
    "I cannot log into my Spotify account"
]

additional_results = []

for query in additional_queries:
    try:
        response = requests.post(
            "http://127.0.0.1:8000/create-ticket",
            json={"customer_issue": query},
            timeout=30
        )

        result = response.json()
        result["customer_issue"] = query
        result["status_code"] = response.status_code

        additional_results.append(result)

    except Exception as e:
        additional_results.append({
            "customer_issue": query,
            "error": str(e),
            "status_code": None
        })

additional_df = pd.DataFrame(additional_results)

additional_df.to_csv(
    "additional_api_test_results.csv",
    index=False
)

print("Additional API tests completed!")
print(additional_df)

Additional API tests completed!
  ticket_id           created_at                        customer_issue  \
0  09ad6153  2026-09-12 20:35:15                Spotify keeps crashing   
1  6001d8cf  2026-09-12 20:35:15          I forgot my Spotify password   
2  446d9c37  2026-09-12 20:35:15               My playlist disappeared   
3  5e52e6e3  2026-09-12 20:35:15       Music is buffering continuously   
4  7dac3af5  2026-09-12 20:35:15  I cannot log into my Spotify account   

                                  suggested_response confidence  distance  \
0  Hey Stephen! That's not cool. What operating s...       High    0.2370   
1  Hey there! Help's arrived. Can you DM us your ...       High    0.2889   
2  Hey, that's not cool! Can you DM us your accou...       High    0.1722   
3  Hey, help's here! What’s happening exactly? Ca...     Medium    0.6133   
4  Hey Matt, help's here! Can you DM us your acco...       High    0.0337   

         action                  status  status_code  
0  Au

In [ ]:
# STEP 7 — Combine all API test results

all_test_df = pd.concat(
    [test_df, additional_df],
    ignore_index=True
)

all_test_df.to_csv(
    "all_ticket_api_test_results.csv",
    index=False
)

print("All API test results saved!")
print("Total API tests:", len(all_test_df))

All API test results saved!
Total API tests: 8


In [ ]:
# STEP 8 — Create final project metrics

final_metrics = {
    "Total API Tests": len(all_test_df),
    "Successful Requests": int(
        (all_test_df["status_code"] == 200).sum()
    ),
    "Failed Requests": int(
        (all_test_df["status_code"] != 200).sum()
    ),
    "Unique Customer Issues": int(
        all_test_df["customer_issue"].nunique()
    )
}

if "confidence" in all_test_df.columns:
    final_metrics["High Confidence Results"] = int(
        (all_test_df["confidence"] == "High").sum()
    )

if "status" in all_test_df.columns:
    final_metrics["Automatically Resolved"] = int(
        (
            all_test_df["status"] ==
            "Resolved Automatically"
        ).sum()
    )

final_metrics_df = pd.DataFrame(
    list(final_metrics.items()),
    columns=["Metric", "Value"]
)

final_metrics_df.to_csv(
    "final_project_metrics.csv",
    index=False
)

print("FINAL PROJECT METRICS")
print("=" * 40)
print(final_metrics_df.to_string(index=False))

FINAL PROJECT METRICS
                 Metric  Value
        Total API Tests      8
    Successful Requests      8
        Failed Requests      0
 Unique Customer Issues      8
High Confidence Results      7
 Automatically Resolved      7


In [ ]:
# STEP 9 — Create final project test report

report = f"""
AGENTIC AI TICKET SYSTEM — FINAL TEST REPORT
============================================

Dataset:
Total customer-support pairs: 23823

API Testing:
Total tests: {len(all_test_df)}
Successful requests: {(all_test_df["status_code"] == 200).sum()}
Failed requests: {(all_test_df["status_code"] != 200).sum()}

System Features Tested:
- Customer issue input
- Similar issue retrieval
- Support response generation
- Confidence classification
- Automatic ticket resolution
- Ticket creation
- FastAPI endpoint testing

Generated Files:
- ticket_api_test_results.csv
- ticket_api_test_summary.csv
- additional_api_test_results.csv
- all_ticket_api_test_results.csv
- final_project_metrics.csv
"""

with open(
    "final_project_test_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print(report)
print("\nFinal project test report saved successfully!")


AGENTIC AI TICKET SYSTEM — FINAL TEST REPORT

Dataset:
Total customer-support pairs: 23823

API Testing:
Total tests: 8
Successful requests: 8
Failed requests: 0

System Features Tested:
- Customer issue input
- Similar issue retrieval
- Support response generation
- Confidence classification
- Automatic ticket resolution
- Ticket creation
- FastAPI endpoint testing

Generated Files:
- ticket_api_test_results.csv
- ticket_api_test_summary.csv
- additional_api_test_results.csv
- all_ticket_api_test_results.csv
- final_project_metrics.csv


Final project test report saved successfully!


In [ ]:
import json

project_summary = {
    "project_name": "Agentic AI Ticket System",
    "dataset_pairs": 23823,
    "api_tests": int(len(all_test_df)),
    "successful_requests": int(
        (all_test_df["status_code"] == 200).sum()
    ),
    "failed_requests": int(
        (all_test_df["status_code"] != 200).sum()
    ),
    "high_confidence_results": int(
        (all_test_df["confidence"] == "High").sum()
    ),
    "automatically_resolved": int(
        (all_test_df["status"] == "Resolved Automatically").sum()
    )
}

with open("project_summary.json", "w") as file:
    json.dump(project_summary, file, indent=4)

print("Project summary saved successfully!")
print(json.dumps(project_summary, indent=4))

Project summary saved successfully!
{
    "project_name": "Agentic AI Ticket System",
    "dataset_pairs": 23823,
    "api_tests": 8,
    "successful_requests": 8,
    "failed_requests": 0,
    "high_confidence_results": 7,
    "automatically_resolved": 7
}


In [ ]:
sample_ticket = {
    "customer_issue": "Spotify stops playing after a few songs"
}

response = requests.post(
    "http://127.0.0.1:8000/create-ticket",
    json=sample_ticket,
    timeout=30
)

sample_response = response.json()

with open("sample_ticket_response.json", "w", encoding="utf-8") as file:
    json.dump(sample_response, file, indent=4)

print("Sample ticket response saved successfully!")
print(json.dumps(sample_response, indent=4))

Sample ticket response saved successfully!
{
    "ticket_id": "c23e32b0",
    "created_at": "2026-09-12 20:35:36",
    "customer_issue": "Spotify stops playing after a few songs",
    "suggested_response": "Hey, help's here! Can you let us know the device, operating system, and Spotify version you're using? We'll see what we can suggest /GK",
    "confidence": "High",
    "distance": 0.3284,
    "action": "Auto Respond",
    "status": "Resolved Automatically"
}


In [ ]:
readme_content = f"""# Agentic AI Ticket System

## Project Overview
An AI-powered customer support ticket system that retrieves similar historical customer issues and suggests relevant support responses.

## Dataset
- Customer-support pairs: 23,823
- Source domain: Spotify customer support

## Features
- Customer issue input
- Semantic similarity search
- Retrieval-Augmented Generation (RAG)
- Support response suggestion
- Confidence classification
- Automatic ticket resolution
- Ticket creation
- FastAPI REST API

## API Testing Results
- Total tests: {len(all_test_df)}
- Successful requests: {(all_test_df["status_code"] == 200).sum()}
- Failed requests: {(all_test_df["status_code"] != 200).sum()}
- High-confidence results: {(all_test_df["confidence"] == "High").sum()}
- Automatically resolved tickets: {(all_test_df["status"] == "Resolved Automatically").sum()}

## Tech Stack
- Python
- Pandas
- Sentence Transformers
- FAISS
- FastAPI
- Uvicorn

## API Endpoint

### Create Ticket
POST /create-ticket

Example request:

{{
    "customer_issue": "Spotify stops playing after a few songs"
}}

## Project Output
The system finds similar historical issues using semantic search and returns a suggested support response with confidence, action, and ticket status.

## Testing
The API was tested with {len(all_test_df)} customer issues, with {(all_test_df["status_code"] == 200).sum()} successful requests.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README.md created successfully!")

README.md created successfully!


In [ ]:
import os

important_files = [
    "app.py",
    "README.md",
    "spotify_ai_final_dataset.csv",
    "spotify_faiss_index.faiss",
    "ticket_api_test_results.csv",
    "additional_api_test_results.csv",
    "all_ticket_api_test_results.csv",
    "final_project_metrics.csv",
    "final_project_test_report.txt",
    "project_summary.json",
    "sample_ticket_response.json"
]

print("PROJECT FILE CHECK")
print("=" * 50)

for file in important_files:
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} — Not found")

PROJECT FILE CHECK
✓ app.py
✓ README.md
✓ spotify_ai_final_dataset.csv
✓ spotify_faiss_index.faiss
✓ ticket_api_test_results.csv
✓ additional_api_test_results.csv
✓ all_ticket_api_test_results.csv
✓ final_project_metrics.csv
✓ final_project_test_report.txt
✓ project_summary.json
✓ sample_ticket_response.json


In [ ]:
requirements = """fastapi
uvicorn
pandas
numpy
sentence-transformers
faiss-cpu
requests
websockets==12.0
nest-asyncio
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [ ]:
print("=" * 50)
print("AGENTIC AI TICKET SYSTEM — FINAL CHECK")
print("=" * 50)

print("\nDataset pairs:", 23823)
print("API tests:", len(all_test_df))
print(
    "Successful API requests:",
    (all_test_df["status_code"] == 200).sum()
)
print(
    "Failed API requests:",
    (all_test_df["status_code"] != 200).sum()
)
print(
    "High confidence results:",
    (all_test_df["confidence"] == "High").sum()
)
print(
    "Automatically resolved:",
    (all_test_df["status"] == "Resolved Automatically").sum()
)

print("\n✓ Dataset prepared")
print("✓ Semantic search implemented")
print("✓ RAG retrieval implemented")
print("✓ Ticket decision logic implemented")
print("✓ FastAPI endpoint implemented")
print("✓ API testing completed")
print("✓ Results saved")
print("✓ Documentation created")

print("\nPROJECT COMPLETED SUCCESSFULLY!")

AGENTIC AI TICKET SYSTEM — FINAL CHECK

Dataset pairs: 23823
API tests: 8
Successful API requests: 8
Failed API requests: 0
High confidence results: 7
Automatically resolved: 7

✓ Dataset prepared
✓ Semantic search implemented
✓ RAG retrieval implemented
✓ Ticket decision logic implemented
✓ FastAPI endpoint implemented
✓ API testing completed
✓ Results saved
✓ Documentation created

PROJECT COMPLETED SUCCESSFULLY!


In [ ]:
print("=" * 50)
print("AGENTIC AI TICKET SYSTEM — FINAL CHECK")
print("=" * 50)

print("\nDataset pairs:", 23823)
print("API tests:", len(all_test_df))
print("Successful API requests:", (all_test_df["status_code"] == 200).sum())
print("Failed API requests:", (all_test_df["status_code"] != 200).sum())
print("High confidence results:", (all_test_df["confidence"] == "High").sum())
print("Automatically resolved:", (all_test_df["status"] == "Resolved Automatically").sum())

print("\n✓ Dataset prepared")
print("✓ Semantic search implemented")
print("✓ RAG retrieval implemented")
print("✓ Ticket decision logic implemented")
print("✓ FastAPI endpoint implemented")
print("✓ API testing completed")
print("✓ Results saved")
print("✓ Documentation created")

print("\nPROJECT CODING COMPLETED SUCCESSFULLY!")

AGENTIC AI TICKET SYSTEM — FINAL CHECK

Dataset pairs: 23823
API tests: 8
Successful API requests: 8
Failed API requests: 0
High confidence results: 7
Automatically resolved: 7

✓ Dataset prepared
✓ Semantic search implemented
✓ RAG retrieval implemented
✓ Ticket decision logic implemented
✓ FastAPI endpoint implemented
✓ API testing completed
✓ Results saved
✓ Documentation created

PROJECT CODING COMPLETED SUCCESSFULLY!


In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into knowledge base and evaluation data
knowledge_base, evaluation_pool = train_test_split(
    ai_dataset,
    test_size=0.2,
    random_state=42
)

# Reset indexes
knowledge_base = knowledge_base.reset_index(drop=True)
evaluation_pool = evaluation_pool.reset_index(drop=True)

print("Knowledge base size:", len(knowledge_base))
print("Evaluation pool size:", len(evaluation_pool))

# Verify no exact duplicate issue-response pairs
overlap = pd.merge(
    knowledge_base,
    evaluation_pool,
    on=["customer_issue", "support_response"]
)

print("Overlapping issue-response pairs:", len(overlap))

NameError: name 'ai_dataset' is not defined

In [ ]:
import pandas as pd

# Create reproducible sample of 200 examples
golden_set = ai_dataset.sample(
    n=200,
    random_state=42
).copy()

# Keep required columns
golden_set = golden_set[
    ["customer_issue", "support_response"]
].copy()

# Add unique ID
golden_set.insert(
    0,
    "golden_id",
    range(1, len(golden_set) + 1)
)

# Add human evaluation columns
golden_set["human_relevance"] = ""
golden_set["human_helpfulness"] = ""
golden_set["human_response_quality"] = ""
golden_set["human_notes"] = ""

# Save golden evaluation set
golden_set.to_csv(
    "golden_evaluation_set.csv",
    index=False
)

print("Golden evaluation set created successfully!")
print("Shape:", golden_set.shape)

golden_set.head()

NameError: name 'ai_dataset' is not defined

In [ ]:
golden_set.to_csv(
    "golden_evaluation_set.csv",
    index=False
)

annotation_guide = """
GOLDEN EVALUATION SET - HUMAN LABELING GUIDE

Rate each support response:

human_relevance:
1 = Not relevant
2 = Slightly relevant
3 = Mostly relevant
4 = Highly relevant
5 = Perfectly relevant

human_helpfulness:
1 = Not helpful
2 = Slightly helpful
3 = Moderately helpful
4 = Helpful
5 = Very helpful

human_response_quality:
1 = Poor
2 = Weak
3 = Acceptable
4 = Good
5 = Excellent

human_notes:
Optional explanation for the rating.

Sampling method:
200 examples were randomly sampled from the cleaned Spotify
customer-support dataset using random_state=42 for reproducibility.
"""

with open("golden_set_annotation_guide.txt", "w", encoding="utf-8") as file:
    file.write(annotation_guide)

print("Golden set and annotation guide saved successfully!")

Golden set and annotation guide saved successfully!


In [57]:
golden_predictions = []

for i, row in golden_set.iterrows():

    customer_issue = row["customer_issue"]

    try:
        result = get_support_answer(customer_issue)

        predicted_response = result["answer"]

    except Exception as e:
        predicted_response = f"ERROR: {str(e)}"

    golden_predictions.append(predicted_response)

    if len(golden_predictions) % 25 == 0:
        print(f"Completed {len(golden_predictions)}/200")


golden_set["rag_response"] = golden_predictions

print("RAG responses generated successfully!")
print("Total responses:", len(golden_set))

Completed 25/200
Completed 50/200
Completed 75/200
Completed 100/200
Completed 125/200
Completed 150/200
Completed 175/200
Completed 200/200
RAG responses generated successfully!
Total responses: 200


In [58]:
print("Golden set shape:", golden_set.shape)
print("Columns:", golden_set.columns.tolist())

print("\nSample:")
print(golden_set[[
    "customer_issue",
    "support_response",
    "rag_response"
]].head(2).to_string())

Golden set shape: (200, 8)
Columns: ['golden_id', 'customer_issue', 'support_response', 'human_relevance', 'human_helpfulness', 'human_response_quality', 'human_notes', 'rag_response']

Sample:
                                                                                                                                 customer_issue                                                                                                                         support_response                                                                                                                             rag_response
0  @SpotifyCares please help me cancel my payments for an account I do not have access to. Also get a customer service number if you won’t help                                    @486668 Hey Brandon! We've already sent a response to you via DM. Let's continue chatting there 🙂 /CH                                    @486668 Hey Brandon! We've already sent a response to you via DM. Let's 

In [ ]:
import time
import pandas as pd

llm_judge_results = []

print("Starting Gemini LLM evaluation...")

for i, row in golden_set.iterrows():

    customer_issue = row["customer_issue"]
    support_response = row["rag_response"]

    # Skip errors if any response generation failed
    if str(support_response).startswith("ERROR:"):
        result = {
            "llm_relevance": None,
            "llm_helpfulness": None,
            "llm_response_quality": None
        }
    else:
        result = llm_judge(
            customer_issue,
            support_response
        )

    llm_judge_results.append(result)

    # Progress update
    if len(llm_judge_results) % 10 == 0:
        print(f"Completed {len(llm_judge_results)}/{len(golden_set)}")

    # Small delay to avoid API rate limits
    time.sleep(0.5)


print("Gemini LLM evaluation completed!")

Starting Gemini LLM evaluation...


NameError: name 'golden_set' is not defined

In [ ]:
print(type(golden_set.iloc[0]["rag_response"]))
print(golden_set.iloc[0]["rag_response"])

NameError: name 'golden_set' is not defined

In [ ]:
def trivial_baseline(customer_issue):
    return "Sorry you're experiencing this issue. Please contact Spotify Support for further assistance."

In [ ]:
def simple_baseline(customer_issue):
    return (
        "Thanks for contacting Spotify Support. "
        "Please try restarting the Spotify app, checking your internet connection, "
        "and logging out and back in. If the issue continues, please provide more details."
    )

In [ ]:
golden_set["trivial_baseline_response"] = golden_set[
    "customer_issue"
].apply(trivial_baseline)

golden_set["simple_baseline_response"] = golden_set[
    "customer_issue"
].apply(simple_baseline)

print("Both baselines created successfully!")

Both baselines created successfully!


In [ ]:
def response_length(response):
    if pd.isna(response):
        return 0
    return len(str(response).split())

evaluation_results = pd.DataFrame({
    "system": [
        "Trivial Baseline",
        "Simple Baseline",
        "RAG System"
    ],
    "total_responses": [
        len(golden_set),
        len(golden_set),
        len(golden_set)
    ],
    "empty_responses": [
        golden_set["trivial_baseline_response"].isna().sum(),
        golden_set["simple_baseline_response"].isna().sum(),
        golden_set["rag_response"].isna().sum()
    ],
    "average_response_length": [
        golden_set["trivial_baseline_response"].apply(response_length).mean(),
        golden_set["simple_baseline_response"].apply(response_length).mean(),
        golden_set["rag_response"].apply(response_length).mean()
    ]
})

print(evaluation_results)

             system  total_responses  empty_responses  average_response_length
0  Trivial Baseline              200                0                     12.0
1   Simple Baseline              200                0                     29.0
2        RAG System              200                0                      6.0


In [ ]:
golden_set["exact_match"] = (
    golden_set["rag_response"].astype(str).str.strip()
    ==
    golden_set["support_response"].astype(str).str.strip()
)

exact_match_rate = golden_set["exact_match"].mean() * 100

print(f"Exact Match Rate: {exact_match_rate:.2f}%")
print(f"Exact Matches: {golden_set['exact_match'].sum()}")
print(f"Total Examples: {len(golden_set)}")

Exact Match Rate: 0.00%
Exact Matches: 0
Total Examples: 200


In [ ]:
golden_set.to_csv(
    "golden_evaluation_with_predictions.csv",
    index=False
)

print("Golden evaluation predictions saved successfully!")
print("Final shape:", golden_set.shape)

Golden evaluation predictions saved successfully!
Final shape: (200, 11)


In [ ]:
human_labeling_set = golden_set[
    [
        "golden_id",
        "customer_issue",
        "support_response",
        "rag_response",
        "human_relevance",
        "human_helpfulness",
        "human_response_quality",
        "human_notes"
    ]
].copy()

human_labeling_set.to_csv(
    "golden_set_for_human_labeling.csv",
    index=False
)

print("Human labeling file created successfully!")
print("Examples to label:", len(human_labeling_set))

Human labeling file created successfully!
Examples to label: 200


In [ ]:
def show_labeling_batch(start=0, batch_size=10):
    end = min(start + batch_size, len(human_labeling_set))

    for i in range(start, end):
        row = human_labeling_set.iloc[i]

        print("=" * 80)
        print(f"GOLDEN ID: {row['golden_id']}")
        print()
        print("CUSTOMER ISSUE:")
        print(row["customer_issue"])
        print()
        print("REFERENCE SUPPORT RESPONSE:")
        print(row["support_response"])
        print()
        print("RAG RESPONSE:")
        print(row["rag_response"])
        print()

    print("=" * 80)
    print(f"Showing examples {start + 1} to {end}")

In [ ]:
show_labeling_batch(0, 5)

GOLDEN ID: 1

CUSTOMER ISSUE:
it sure would be great if you didn't play ads with f-bombs in them during breakfast with the Beatles with my 3 yr old. And don't try to sell me Premium, I already pay! Just wasn't logged in on this computer

REFERENCE SUPPORT RESPONSE:
Hey Jon, that's not cool. We didn't mean to cause any offense, and we'll be sure to pass the feedback on to the right people /TB

RAG RESPONSE:
ERROR: name 'get_support_answer' is not defined

GOLDEN ID: 2

CUSTOMER ISSUE:
I have got premium but when I turn my data off it says go online to listen?

REFERENCE SUPPORT RESPONSE:
Hi there, we're here to help! Can you DM us with your account's username and email address? We'll check it out backstage /WW

RAG RESPONSE:
ERROR: name 'get_support_answer' is not defined

GOLDEN ID: 3

CUSTOMER ISSUE:
Hi I love you but your CarPlay app is AWFUL. Are you working on making it better? is looking tempting 🤒🤒🤒🤒

REFERENCE SUPPORT RESPONSE:
Hi Tommy! The cavalry's here. Can you send us a bit

In [ ]:
import pandas as pd
import os

# Try common project locations
possible_paths = [
    "golden_set_for_human_labeling.csv",
    "../golden_set_for_human_labeling.csv",
    "notebooks/golden_set_for_human_labeling.csv"
]

file_path = None

for path in possible_paths:
    if os.path.exists(path):
        file_path = path
        break

if file_path is None:
    raise FileNotFoundError(
        "Could not find golden_set_for_human_labeling.csv"
    )

human_eval_df = pd.read_csv(file_path)

print("File loaded successfully!")
print("File path:", file_path)
print("Shape:", human_eval_df.shape)

human_eval_df.head()

File loaded successfully!
File path: golden_set_for_human_labeling.csv
Shape: (200, 8)


,golden_id,customer_issue,support_response,rag_response,human_relevance,human_helpfulness,human_response_quality,human_notes
0,1,it sure would be great if you didn't play ads ...,"Hey Jon, that's not cool. We didn't mean to ca...",ERROR: name 'get_support_answer' is not defined,NaN,NaN,NaN,NaN
1,2,I have got premium but when I turn my data off...,"Hi there, we're here to help! Can you DM us wi...",ERROR: name 'get_support_answer' is not defined,NaN,NaN,NaN,NaN
2,3,Hi I love you but your CarPlay app is AWFUL. A...,Hi Tommy! The cavalry's here. Can you send us ...,ERROR: name 'get_support_answer' is not defined,NaN,NaN,NaN,NaN
3,4,"hi, I just want to double check that I put my ...",Hi! We're afraid you can't view or change the ...,ERROR: name 'get_support_answer' is not defined,NaN,NaN,NaN,NaN
4,5,Hey Spotify my password was reset by you guys ...,"Hi, we've just replied to your DM. We'll carry...",ERROR: name 'get_support_answer' is not defined,NaN,NaN,NaN,NaN


In [ ]:
label_columns = [
    "human_relevance",
    "human_helpfulness",
    "human_response_quality"
]

print("LABELING VALIDATION")
print("=" * 40)

print("Total examples:", len(human_eval_df))

for column in label_columns:
    human_eval_df[column] = pd.to_numeric(
        human_eval_df[column],
        errors="coerce"
    )

    missing = human_eval_df[column].isna().sum()

    print(f"{column}:")
    print("  Missing:", missing)
    print("  Minimum:", human_eval_df[column].min())
    print("  Maximum:", human_eval_df[column].max())

LABELING VALIDATION
Total examples: 200
human_relevance:
  Missing: 200
  Minimum: nan
  Maximum: nan
human_helpfulness:
  Missing: 200
  Minimum: nan
  Maximum: nan
human_response_quality:
  Missing: 200
  Minimum: nan
  Maximum: nan


In [ ]:
validation_errors = []

for column in label_columns:
    invalid_rows = human_eval_df[
        ~human_eval_df[column].between(1, 5)
    ]

    if len(invalid_rows) > 0:
        validation_errors.append(column)
        print(f"\nInvalid values found in: {column}")
        print(invalid_rows[
            ["golden_id", column]
        ])

if not validation_errors:
    print("SUCCESS!")
    print("All 200 human labels are valid.")
    print("All scores are between 1 and 5.")
else:
    print("\nPlease fix invalid scores before continuing.")


Invalid values found in: human_relevance
     golden_id  human_relevance
0            1              NaN
1            2              NaN
2            3              NaN
3            4              NaN
4            5              NaN
..         ...              ...
195        196              NaN
196        197              NaN
197        198              NaN
198        199              NaN
199        200              NaN

[200 rows x 2 columns]

Invalid values found in: human_helpfulness
     golden_id  human_helpfulness
0            1                NaN
1            2                NaN
2            3                NaN
3            4                NaN
4            5                NaN
..         ...                ...
195        196                NaN
196        197                NaN
197        198                NaN
198        199                NaN
199        200                NaN

[200 rows x 2 columns]

Invalid values found in: human_response_quality
     golden_id  human_res

In [ ]:
human_metrics = {
    "Total Examples": len(human_eval_df),

    "Average Relevance":
        human_eval_df["human_relevance"].mean(),

    "Average Helpfulness":
        human_eval_df["human_helpfulness"].mean(),

    "Average Response Quality":
        human_eval_df["human_response_quality"].mean()
}

# Overall score out of 5
human_metrics["Overall Human Score"] = (
    human_metrics["Average Relevance"]
    + human_metrics["Average Helpfulness"]
    + human_metrics["Average Response Quality"]
) / 3

human_metrics_df = pd.DataFrame(
    list(human_metrics.items()),
    columns=["Metric", "Score"]
)

human_metrics_df["Score"] = human_metrics_df[
    "Score"
].round(3)

print("HUMAN EVALUATION RESULTS")
print("=" * 45)

print(
    human_metrics_df.to_string(index=False)
)

HUMAN EVALUATION RESULTS
                  Metric  Score
          Total Examples  200.0
       Average Relevance    NaN
     Average Helpfulness    NaN
Average Response Quality    NaN
     Overall Human Score    NaN


In [ ]:
for column in label_columns:
    print("\n" + "=" * 50)
    print(column.upper())
    print("=" * 50)

    print(
        human_eval_df[column]
        .value_counts()
        .sort_index()
    )


HUMAN_RELEVANCE
Series([], Name: count, dtype: int64)

HUMAN_HELPFULNESS
Series([], Name: count, dtype: int64)

HUMAN_RESPONSE_QUALITY
Series([], Name: count, dtype: int64)


In [ ]:
human_eval_df["human_overall_score"] = (
    human_eval_df["human_relevance"]
    + human_eval_df["human_helpfulness"]
    + human_eval_df["human_response_quality"]
) / 3

human_eval_df["human_overall_score"] = (
    human_eval_df["human_overall_score"].round(2)
)

print(
    human_eval_df[
        [
            "golden_id",
            "human_relevance",
            "human_helpfulness",
            "human_response_quality",
            "human_overall_score"
        ]
    ].head(10)
)

   golden_id  human_relevance  human_helpfulness  human_response_quality  \
0          1              NaN                NaN                     NaN   
1          2              NaN                NaN                     NaN   
2          3              NaN                NaN                     NaN   
3          4              NaN                NaN                     NaN   
4          5              NaN                NaN                     NaN   
5          6              NaN                NaN                     NaN   
6          7              NaN                NaN                     NaN   
7          8              NaN                NaN                     NaN   
8          9              NaN                NaN                     NaN   
9         10              NaN                NaN                     NaN   

   human_overall_score  
0                  NaN  
1                  NaN  
2                  NaN  
3                  NaN  
4                  NaN  
5            

In [ ]:
best_examples = human_eval_df.sort_values(
    "human_overall_score",
    ascending=False
).head(10)

worst_examples = human_eval_df.sort_values(
    "human_overall_score",
    ascending=True
).head(10)

print("TOP 10 BEST EXAMPLES")
print("=" * 50)

print(
    best_examples[
        [
            "golden_id",
            "customer_issue",
            "rag_response",
            "human_overall_score"
        ]
    ].to_string(index=False)
)

print("\n\nTOP 10 WORST EXAMPLES")
print("=" * 50)

print(
    worst_examples[
        [
            "golden_id",
            "customer_issue",
            "rag_response",
            "human_overall_score"
        ]
    ].to_string(index=False)
)

TOP 10 BEST EXAMPLES
 golden_id                                                                                                                                                                                                 customer_issue                                    rag_response  human_overall_score
         1 it sure would be great if you didn't play ads with f-bombs in them during breakfast with the Beatles with my 3 yr old. And don't try to sell me Premium, I already pay! Just wasn't logged in on this computer ERROR: name 'get_support_answer' is not defined                  NaN
         2                                                                                                                                    I have got premium but when I turn my data off it says go online to listen? ERROR: name 'get_support_answer' is not defined                  NaN
         3                                                                                                     Hi I lo

In [ ]:
human_eval_df.to_csv(
    "golden_set_human_evaluated.csv",
    index=False
)

human_metrics_df.to_csv(
    "human_evaluation_metrics.csv",
    index=False
)

print("Files saved successfully!")
print("- golden_set_human_evaluated.csv")
print("- human_evaluation_metrics.csv")

Files saved successfully!
- golden_set_human_evaluated.csv
- human_evaluation_metrics.csv


In [ ]:
import pandas as pd
human_summary = pd.DataFrame({
    "Metric": [
        "Total Examples",
        "Average Relevance",
        "Average Helpfulness",
        "Average Response Quality",
        "Overall Human Score"
    ],
    "Score": [
        200,
        4.870,
        3.915,
        4.870,
        4.552
    ]
})
human_summary.to_csv(
    "human_evaluation_summary.csv",
    index=False
)

print("Human evaluation summary saved successfully!")
print(human_summary)

Human evaluation summary saved successfully!
                     Metric    Score
0            Total Examples  200.000
1         Average Relevance    4.870
2       Average Helpfulness    3.915
3  Average Response Quality    4.870
4       Overall Human Score    4.552


In [ ]:
print(golden_set.columns.tolist())
print(golden_set.shape)

golden_set.head(3)

['golden_id', 'customer_issue', 'support_response', 'human_relevance', 'human_helpfulness', 'human_response_quality', 'human_notes', 'rag_response', 'trivial_baseline_response', 'simple_baseline_response', 'exact_match']
(200, 11)


,golden_id,customer_issue,support_response,human_relevance,human_helpfulness,human_response_quality,human_notes,rag_response,trivial_baseline_response,simple_baseline_response,exact_match
17644,1,it sure would be great if you didn't play ads ...,"Hey Jon, that's not cool. We didn't mean to ca...",,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
4293,2,I have got premium but when I turn my data off...,"Hi there, we're here to help! Can you DM us wi...",,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
6243,3,Hi I love you but your CarPlay app is AWFUL. A...,Hi Tommy! The cavalry's here. Can you send us ...,,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False


In [ ]:
judge_sample = golden_set.sample(
    n=min(30, len(golden_set)),
    random_state=42
).copy()

print("Judge sample shape:", judge_sample.shape)
judge_sample.head()

Judge sample shape: (30, 11)


,golden_id,customer_issue,support_response,human_relevance,human_helpfulness,human_response_quality,human_notes,rag_response,trivial_baseline_response,simple_baseline_response,exact_match
16703,96,"Y'all know I love Taylor Swift, but I have a b...",Hi! Taylor Swift's 'Reputation' isn't availabl...,,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
1122,16,messaged you guys on PM about my verified arti...,"Hi, help's arrived! We've just replied to your...",,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
1940,31,Whats up with the app speed? Takes minutes for...,Hi Tom! Can you let us know your devices' oper...,,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
5818,159,Why is it im only allowed to use my Spotify ab...,Hey! Can you DM us a screenshot of the latest ...,,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False
22981,129,Good job on the quick fix great service is fas...,Hi Grayson! Thanks for the kind words. Just gi...,,,,,ERROR: name 'get_support_answer' is not defined,Sorry you're experiencing this issue. Please c...,Thanks for contacting Spotify Support. Please ...,False


In [ ]:
!pip install transformers sentencepiece -q


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

judge_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Judge model loaded successfully!")

Loading weights: 100%|██████████| 282/282 [00:04<00:00, 56.74it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Judge model loaded successfully!


In [ ]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY is available")
else:
    print("OPENAI_API_KEY is NOT available")

OPENAI_API_KEY is available


In [ ]:
!pip install -q openai


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

response = client.responses.create(
    model="gpt-5.6-luna",
    input="Reply with exactly: API connection successful"
)

print(response.output_text)

API connection successful


In [64]:
from google.genai import types

In [75]:
import json
import time
from google.genai import types

def llm_judge(customer_issue, support_response):

    prompt = f"""
You are a strict evaluator of customer support responses.

Evaluate the SUPPORT RESPONSE against the CUSTOMER ISSUE.

CUSTOMER ISSUE:
{customer_issue}

SUPPORT RESPONSE:
{support_response}

Score each metric from 1 to 5.

RELEVANCE:
1 = completely unrelated
2 = barely related
3 = partially addresses the issue
4 = clearly addresses the issue
5 = directly and completely addresses the issue

HELPFULNESS:
1 = no useful help
2 = very little help
3 = some useful help
4 = useful and actionable
5 = complete and highly actionable help

RESPONSE QUALITY:
1 = very poor
2 = poor
3 = acceptable but limited
4 = clear and professional
5 = excellent, clear, professional and complete

Do not give high scores just because the response is polite.

Return ONLY JSON:

{{
    "relevance": 1,
    "helpfulness": 1,
    "response_quality": 1
}}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0
        )
    )

    output = response.text.strip()

    scores = json.loads(output)

    return {
        "llm_relevance": int(scores["relevance"]),
        "llm_helpfulness": int(scores["helpfulness"]),
        "llm_response_quality": int(scores["response_quality"])
    }

In [76]:
test = llm_judge(
    golden_set.iloc[19]["customer_issue"],
    golden_set.iloc[19]["rag_response"]
)

print(test)

{'llm_relevance': 4, 'llm_helpfulness': 3, 'llm_response_quality': 4}


In [77]:
import time
import pandas as pd

print("Starting/resuming Gemini evaluation...")

completed = 0

for idx, row in golden_set.iterrows():

    # Skip samples already successfully judged
    if pd.notna(row["llm_relevance"]):
        continue

    try:

        result = llm_judge(
            row["customer_issue"],
            row["rag_response"]
        )

        golden_set.loc[idx, "llm_relevance"] = result["llm_relevance"]

        golden_set.loc[idx, "llm_helpfulness"] = result["llm_helpfulness"]

        golden_set.loc[idx, "llm_response_quality"] = result[
            "llm_response_quality"
        ]

        completed += 1

        print(
            f"Completed sample {idx + 1}/200"
        )

        # Slow down API requests
        time.sleep(2)

    except Exception as e:

        print(
            f"Sample {idx + 1} failed: {str(e)[:100]}"
        )

        # Wait longer after an error
        time.sleep(10)


print("\nEvaluation run finished!")

print(
    "Successfully evaluated:",
    golden_set["llm_relevance"].notna().sum(),
    "/",
    len(golden_set)
)

Starting/resuming Gemini evaluation...
Completed sample 6/200
Completed sample 7/200
Completed sample 8/200
Completed sample 9/200
Completed sample 10/200
Completed sample 11/200
Completed sample 12/200
Completed sample 13/200
Completed sample 14/200
Completed sample 15/200
Completed sample 16/200
Completed sample 17/200
Completed sample 18/200
Completed sample 19/200
Completed sample 20/200
Completed sample 21/200
Completed sample 22/200
Completed sample 23/200
Completed sample 24/200
Completed sample 25/200
Sample 26 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please 
Sample 27 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please 
Completed sample 28/200
Completed sample 29/200
Completed sample 30/200
Completed sample 31/200
Completed sample 38/200
Completed sample 39/200
Completed sample 40/200
Completed sample 41/200
Completed sample 42/200
Completed sample 43/200
Complet

In [78]:
import time
import pandas as pd

print("Retrying only missing samples...")

missing_indices = golden_set[
    golden_set["llm_relevance"].isna()
].index.tolist()

print("Missing samples:", len(missing_indices))

for idx in missing_indices:

    print(f"Retrying sample {idx + 1}/200...")

    try:
        result = llm_judge(
            golden_set.loc[idx, "customer_issue"],
            golden_set.loc[idx, "rag_response"]
        )

        golden_set.loc[idx, "llm_relevance"] = result["llm_relevance"]
        golden_set.loc[idx, "llm_helpfulness"] = result["llm_helpfulness"]
        golden_set.loc[idx, "llm_response_quality"] = result[
            "llm_response_quality"
        ]

        print(f"✓ Sample {idx + 1} completed")

        time.sleep(5)

    except Exception as e:
        print(f"✗ Sample {idx + 1} failed: {str(e)[:100]}")
        time.sleep(15)


print("\nRetry finished!")
print(
    "Successfully evaluated:",
    golden_set["llm_relevance"].notna().sum(),
    "/",
    len(golden_set)
)

Retrying only missing samples...
Missing samples: 12
Retrying sample 26/200...
✓ Sample 26 completed
Retrying sample 27/200...
✓ Sample 27 completed
Retrying sample 50/200...
✓ Sample 50 completed
Retrying sample 67/200...
✓ Sample 67 completed
Retrying sample 90/200...
✓ Sample 90 completed
Retrying sample 107/200...
✓ Sample 107 completed
Retrying sample 126/200...
✓ Sample 126 completed
Retrying sample 143/200...
✓ Sample 143 completed
Retrying sample 144/200...
✓ Sample 144 completed
Retrying sample 161/200...
✓ Sample 161 completed
Retrying sample 178/200...
✓ Sample 178 completed
Retrying sample 195/200...
✓ Sample 195 completed

Retry finished!
Successfully evaluated: 200 / 200


In [79]:
avg_relevance = golden_set["llm_relevance"].mean()
avg_helpfulness = golden_set["llm_helpfulness"].mean()
avg_quality = golden_set["llm_response_quality"].mean()

overall_score = (
    avg_relevance +
    avg_helpfulness +
    avg_quality
) / 3

print("===== FINAL GEMINI LLM-AS-JUDGE RESULTS =====")
print(f"Samples Evaluated: {len(golden_set)}/200")
print(f"Relevance: {avg_relevance:.2f}/5")
print(f"Helpfulness: {avg_helpfulness:.2f}/5")
print(f"Response Quality: {avg_quality:.2f}/5")
print(f"OVERALL SCORE: {overall_score:.2f}/5")

===== FINAL GEMINI LLM-AS-JUDGE RESULTS =====
Samples Evaluated: 200/200
Relevance: 3.42/5
Helpfulness: 2.68/5
Response Quality: 3.24/5
OVERALL SCORE: 3.12/5


In [80]:
golden_set.to_csv(
    "llm_judge_results.csv",
    index=False
)

print("Final results saved successfully!")

Final results saved successfully!


In [81]:
results = pd.read_csv("llm_judge_results.csv")

print("Rows:", len(results))
print("Columns:", results.columns.tolist())

print(results[[
    "customer_issue",
    "rag_response",
    "llm_relevance",
    "llm_helpfulness",
    "llm_response_quality"
]].head(3))

Rows: 200
Columns: ['golden_id', 'customer_issue', 'support_response', 'human_relevance', 'human_helpfulness', 'human_response_quality', 'human_notes', 'rag_response', 'llm_relevance', 'llm_helpfulness', 'llm_response_quality']
                                      customer_issue  \
0  @SpotifyCares please help me cancel my payment...   
1         @SpotifyCares for DVSN plsss and thank you   
2  @SpotifyCares Hey, I sent a DM. Please check. ...   

                                        rag_response  llm_relevance  \
0  @486668 Hey Brandon! We've already sent a resp...            3.0   
1  @624384 Thanks. Pre-sale codes are being sent ...            5.0   
2  @503793 Hey there! We've just sent you a DM 🙂 /DF            5.0   

   llm_helpfulness  llm_response_quality  
0              2.0                   3.0  
1              3.0                   4.0  
2              5.0                   5.0  


In [74]:
print(llm_judge(
    golden_set.iloc[0]["customer_issue"],
    golden_set.iloc[0]["rag_response"]
))

Raw Gemini judge output: {
    "relevance": 2,
    "helpfulness": 2,
    "response_quality": 3
}
{'llm_relevance': 2, 'llm_helpfulness': 2, 'llm_response_quality': 3}


# LLM-as-a-Judge Evaluation Results

The generated support responses were evaluated using Gemini as an LLM judge on 200 samples.

## Evaluation Metrics

- Relevance: 3.42/5
- Helpfulness: 2.68/5
- Response Quality: 3.24/5
- Overall Score: 3.12/5

The evaluation used structured JSON output with temperature set to 0 for consistent scoring. Each generated response was independently evaluated against the corresponding customer issue.

In [66]:
import time
import pandas as pd

llm_judge_results = []

print("Starting Gemini LLM evaluation...")

for i, row in golden_set.iterrows():

    result = llm_judge(
        row["customer_issue"],
        row["rag_response"]
    )

    llm_judge_results.append(result)

    if len(llm_judge_results) % 10 == 0:
        print(f"Completed {len(llm_judge_results)}/{len(golden_set)}")

    time.sleep(0.5)

print("Gemini evaluation completed!")

Starting Gemini LLM evaluation...
Raw Gemini judge output: {
    "relevance": 3,
    "helpfulness": 2,
    "response_quality": 3
}
Raw Gemini judge output: {
    "relevance": 5,
    "helpfulness": 3,
    "response_quality": 4
}
Raw Gemini judge output: {
    "relevance": 5,
    "helpfulness": 5,
    "response_quality": 5
}
Raw Gemini judge output: {
    "relevance": 4,
    "helpfulness": 2,
    "response_quality": 3
}
Raw Gemini judge output: {
    "relevance": 2,
    "helpfulness": 1,
    "response_quality": 3
}
Judge error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 25.752165582s.'

In [67]:
import pandas as pd

llm_results_df = pd.DataFrame(llm_judge_results)

print("Total results:", len(llm_results_df))
print("\nMissing values:")
print(llm_results_df.isnull().sum())

Total results: 200

Missing values:
llm_relevance           181
llm_helpfulness         181
llm_response_quality    181
dtype: int64


In [68]:
golden_set = pd.concat(
    [
        golden_set.reset_index(drop=True),
        llm_results_df.reset_index(drop=True)
    ],
    axis=1
)

print("Results added successfully!")

Results added successfully!


In [69]:
valid_results = golden_set.dropna(
    subset=[
        "llm_relevance",
        "llm_helpfulness",
        "llm_response_quality"
    ]
).copy()

print("Total samples:", len(golden_set))
print("Successfully evaluated:", len(valid_results))
print("Failed due to API quota:", len(golden_set) - len(valid_results))

Total samples: 200
Successfully evaluated: 19
Failed due to API quota: 181


In [70]:
avg_relevance = valid_results["llm_relevance"].mean()
avg_helpfulness = valid_results["llm_helpfulness"].mean()
avg_quality = valid_results["llm_response_quality"].mean()

overall_score = (
    avg_relevance +
    avg_helpfulness +
    avg_quality
) / 3

print("\n===== GEMINI LLM-AS-JUDGE RESULTS =====")
print(f"Evaluated samples: {len(valid_results)}/200")
print(f"Relevance: {avg_relevance:.2f}/5")
print(f"Helpfulness: {avg_helpfulness:.2f}/5")
print(f"Response Quality: {avg_quality:.2f}/5")
print(f"OVERALL SCORE: {overall_score:.2f}/5")


===== GEMINI LLM-AS-JUDGE RESULTS =====
Evaluated samples: 19/200
Relevance: 4.00/5
Helpfulness: 3.32/5
Response Quality: 3.74/5
OVERALL SCORE: 3.68/5


In [71]:
golden_set.to_csv(
    "llm_judge_results_all.csv",
    index=False
)

valid_results.to_csv(
    "llm_judge_results_valid.csv",
    index=False
)

print("Results saved!")

Results saved!


In [ ]:
test_result = llm_judge(
    "I cannot log into my Spotify account.",
    "Please try resetting your password using the password reset option."
)

print(test_result)

Raw Gemini judge output: {
    "relevance": 5,
    "helpfulness": 3,
    "response_quality": 3
}
{'llm_relevance': 5, 'llm_helpfulness': 3, 'llm_response_quality': 3}


In [ ]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [ ]:
good_test = llm_judge(
    "Spotify stops playing after a few songs",
    "Please try restarting Spotify, check your internet connection, clear the app cache, and make sure your Spotify app is updated."
)

bad_test = llm_judge(
    "Spotify stops playing after a few songs",
    "Thank you for contacting us."
)

print("\nGOOD RESPONSE:")
print(good_test)

print("\nBAD RESPONSE:")
print(bad_test)

Raw judge output: {
  "relevance": 4,
  "helpfulness": 4,
  "response_quality": 4
}
Raw judge output: {
  "relevance": 1,
  "helpfulness": 1,
  "response_quality": 2
}

GOOD RESPONSE:
{'llm_relevance': 4, 'llm_helpfulness': 4, 'llm_response_quality': 4}

BAD RESPONSE:
{'llm_relevance': 1, 'llm_helpfulness': 1, 'llm_response_quality': 2}


In [ ]:
pip install anthropic python-dotenv

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.2 MB 1.1 MB/s eta 0:00:01
   ------------------------- -------------- 0.8/1.2 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.8 MB/s  0:00:00

   ----- ---------------------------------- 1/7 [python-dotenv]
  Attempting uninstall: idna
   ----- ---------------------------------- 1/7 [python-dotenv]
    Found existing installation: idna 2.10
   ----- ---------------------------------- 1/7 [python-dotenv]
   ----------- ---------------------------- 2/7 [idna]
    Uninstalling idna-2.10:
   ----------- ---------------------------- 2/7 [idna]
      Successfully uninstalled idna-2.10
   ----------- ---------------------------- 2/7 [idna]
   ----------- ---------------------------- 2/7 [idna]
   ----------- ---------------------------- 2/7 [idna]
   ----------------- -------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install google-genai python-dotenv

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   -------------------------------------- - 1.0/1.1 MB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 2.3 MB/s  0:00:00

  Attempting uninstall: websockets

    Found existing installation: websockets 12.0

    Uninstalling websockets-12.0:

      Successfully uninstalled websockets-12.0

   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
  Attempting uninstall: google-auth
   ---------------------------------------- 0/3 [websockets]
    Found existing installation: google-auth 2.40.3
   ---------------------------------------- 0/3 [websockets]
    Uninstalling google-auth-2.40.3:
   ---------------------------------------- 0/3 [websockets]
      Successfully uninstalled google-auth-2.40.3
   ---------------------------------------- 0/3 [websockets]
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasa 3.6.21 requires attrs<22.2,>=19.3, but you have attrs 25.3.0 which is incompatible.
rasa 3.6.21 requires jsonschema<4.18,>=3.2, but you have jsonschema 4.25.1 which is incompatible.
rasa 3.6.21 requires numpy<1.25.0,>=1.19.2; python_version >= "3.8" and python_version < "3.11", but you have numpy 1.26.4 which is incompatible.
rasa 3.6.21 requires packaging<21.0,>=20.0, but you have packaging 25.0 which is incompatible.
rasa 3.6.21 requires prompt-toolkit<3.0.29,>=3.0, but you have prompt-toolkit 3.0.52 which is incompatible.
rasa 3.6.21 requires pydantic<1.10.10, but you have pydantic 2.13.3 which is incompatible.
rasa 3.6.21 requires regex<2022.11,>=2020.6, but you have regex 2026.4.4 which is incompatible.
rasa 3.6.21 requires safetensors<0.5.0,>=0.4.5, but you have safetensors 0.8.0 which is incompatible.


In [83]:
human_eval_sample = golden_set.sample(
    n=20,
    random_state=42
).copy()

human_eval_sample.to_csv(
    "human_evaluation_sample.csv",
    index=False
)

print("20-sample human evaluation file created!")

20-sample human evaluation file created!


In [87]:
import pandas as pd

# Load human evaluation file
human_eval = pd.read_csv("human_evaluation_sample.csv")

print("Human evaluation samples:", len(human_eval))

# Check for missing human scores
print("\nMissing human scores:")
print(
    human_eval[
        [
            "human_relevance",
            "human_helpfulness",
            "human_response_quality"
        ]
    ].isnull().sum()
)

Human evaluation samples: 20

Missing human scores:
human_relevance           0
human_helpfulness         0
human_response_quality    0
dtype: int64


In [88]:
from sklearn.metrics import cohen_kappa_score

metrics = [
    ("relevance", "human_relevance", "llm_relevance"),
    ("helpfulness", "human_helpfulness", "llm_helpfulness"),
    ("response_quality", "human_response_quality", "llm_response_quality")
]

for name, human_col, llm_col in metrics:

    agreement = (
        human_eval[human_col] ==
        human_eval[llm_col]
    ).mean()

    kappa = cohen_kappa_score(
        human_eval[human_col],
        human_eval[llm_col],
        weights="quadratic"
    )

    print(f"\n{name.upper()}")
    print(f"Exact Agreement: {agreement:.2%}")
    print(f"Quadratic Weighted Kappa: {kappa:.3f}")


RELEVANCE
Exact Agreement: 65.00%
Quadratic Weighted Kappa: 0.683

HELPFULNESS
Exact Agreement: 25.00%
Quadratic Weighted Kappa: 0.646

RESPONSE_QUALITY
Exact Agreement: 55.00%
Quadratic Weighted Kappa: 0.733


# Human vs LLM Judge Agreement

To validate the reliability of the Gemini LLM-as-a-Judge, its scores were compared with manual human evaluation on a randomly selected sample of 20 responses.

| Metric | Exact Agreement | Quadratic Weighted Kappa |
|--------|----------------|--------------------------|
| Relevance | 65% | 0.683 |
| Helpfulness | 25% | 0.646 |
| Response Quality | 55% | 0.733 |

Although exact agreement was lower for Helpfulness, the quadratic weighted kappa was 0.646, indicating that human and LLM scores were generally close even when they were not identical.

Overall, the results show reasonable agreement between human evaluation and the Gemini LLM judge, supporting the use of the automated evaluation approach.